#  Kayfa Student Analytics — Day 1: Load, Clean, Join
## Data Analytics Evaluation ·

> **Goal:** End today with a clean, joined master DataFrame and a cleaning log with **30+ of the 36** planted problems documented.
>


---
# ═══════════════════════════════════════════
# SECTION 0 · SETUP & IMPORTS
# ═══════════════════════════════════════════

In [ ]:
import os

os.makedirs("cleaned_data", exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)

# Collect every problem here automatically
cleaning_log = []

def log_problem(problem_num, file, description, decision):
    """Log a data quality problem."""
    cleaning_log.append({
        'Problem #': problem_num,
        'File(s)': file,
        'Description': description,
        'Decision': decision
    })
    print(f"\U0001f534 Problem #{problem_num}: {description}")

problem_counter = [0]

def next_problem():
    problem_counter[0] += 1
    return problem_counter[0]

print("\u2705 Setup complete")

✅ Setup complete


---
# ═══════════════════════════════════════════
# SECTION 1 · LOAD ALL 8 FILES
# ═══════════════════════════════════════════


In [ ]:
import zipfile
import os

zip_path = '/content/student_edu_info.zip'
extract_path = '/content/student_data/'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

# See what's inside
for root, dirs, files in os.walk(extract_path):
    for f in files:
        print(os.path.join(root, f))
DATA_PATH = '/content/student_data/student_edu_info/'

/content/student_data/student_edu_info/courses.csv
/content/student_data/student_edu_info/students.csv
/content/student_data/student_edu_info/assignment_submissions.csv
/content/student_data/student_edu_info/grades.json
/content/student_data/student_edu_info/groups.csv
/content/student_data/student_edu_info/attendance.xlsx
/content/student_data/student_edu_info/engagement_events.csv
/content/student_data/student_edu_info/concepts_performance.csv


## 1A · Load CSVs

In [ ]:
courses = pd.read_csv(f'{DATA_PATH}courses.csv')
groups = pd.read_csv(f'{DATA_PATH}groups.csv')
students = pd.read_csv(f'{DATA_PATH}students.csv')
concepts = pd.read_csv(f'{DATA_PATH}concepts_performance.csv')
engagement = pd.read_csv(f'{DATA_PATH}engagement_events.csv')
submissions = pd.read_csv(f'{DATA_PATH}assignment_submissions.csv')

print("=== CSV FILES LOADED ===")
for name, df in [('courses', courses), ('groups', groups), ('students', students),
                  ('concepts', concepts), ('engagement', engagement), ('submissions', submissions)]:
    print(f"\n\U0001f4c4 {name}: {df.shape[0]} rows x {df.shape[1]} cols")
    print(f"   Columns: {list(df.columns)}")

=== CSV FILES LOADED ===

📄 courses: 7 rows x 8 cols
   Columns: ['course_id', 'course_name', 'category', 'difficulty_level', 'duration_weeks', 'short_description', 'modules', 'is_active']

📄 groups: 12 rows x 7 cols
   Columns: ['group_id', 'group_name', 'course_id', 'stated_num_students', 'session_day', 'session_time', 'instructor']

📄 students: 502 rows x 8 cols
   Columns: ['student_id', 'full_name', 'age', 'gender', 'city', 'email', 'group_id', 'enrollment_date']

📄 concepts: 12008 rows x 10 cols
   Columns: ['record_id', 'student_id', 'course_id', 'assessment_id', 'question_no', 'concept_id', 'concept_name', 'score_pct', 'mastery_status', 'timestamp']

📄 engagement: 30866 rows x 6 cols
   Columns: ['event_id', 'student_id', 'event_type', 'event_datetime', 'duration_seconds', 'device']

📄 submissions: 1504 rows x 9 cols
   Columns: ['submission_id', 'student_id', 'course_id', 'assessment_id', 'deadline', 'submitted_at', 'is_late', 'time_spent_minutes', 'attempts']


## 1B · Load & Flatten JSON (grades)

In [ ]:
with open(f'{DATA_PATH}grades.json', 'r') as f:
    grades_raw = json.load(f)

print("Type:", type(grades_raw))
if isinstance(grades_raw, list):
    print("Length:", len(grades_raw))
    print("First record keys:", grades_raw[0].keys())
    print("\nFirst record sample:")
    print(json.dumps(grades_raw[0], indent=2)[:500])

Type: <class 'list'>
Length: 500
First record keys: dict_keys(['student_id', 'course_id', 'group_id', 'grades'])

First record sample:
{
  "student_id": "S0001",
  "course_id": "C002",
  "group_id": "G03",
  "grades": [
    {
      "grade_id": "GR00001",
      "assessment_id": "C002-QZ",
      "assessment_title": "Quiz 1",
      "type": "quiz",
      "score": -10,
      "max_score": 100,
      "date": "2025-12-27"
    },
    {
      "grade_id": "GR00002",
      "assessment_id": "C002-QZ",
      "assessment_title": "Quiz 2",
      "type": "quiz",
      "score": 69.1,
      "max_score": 100,
      "date": "2026-01-10"
    },
    


In [ ]:
# Flatten nested grades array
# Adjust field names based on what you see above

grades_rows = []
for record in grades_raw:
    student_id = record.get('student_id')
    course_id = record.get('course_id')
    group_id = record.get('group_id')

    for g in record.get('grades', []):
        row = {
            'student_id': student_id,
            'course_id': course_id,
            'group_id': group_id,
            'assessment_id': g.get('assessment_id'),
            'assessment_title': g.get('assessment_title'),
            'type': g.get('type'),
            'score': g.get('score'),
            'max_score': g.get('max_score'),
            'date': g.get('date')
        }
        grades_rows.append(row)

grades = pd.DataFrame(grades_rows)
print(f"\U0001f4c4 grades: {grades.shape[0]} rows x {grades.shape[1]} cols")
print(f"   Columns: {list(grades.columns)}")
grades.head()

📄 grades: 5502 rows x 9 cols
   Columns: ['student_id', 'course_id', 'group_id', 'assessment_id', 'assessment_title', 'type', 'score', 'max_score', 'date']


,student_id,course_id,group_id,assessment_id,assessment_title,type,score,max_score,date
0,S0001,C002,G03,C002-QZ,Quiz 1,quiz,-10.0,100,2025-12-27
1,S0001,C002,G03,C002-QZ,Quiz 2,quiz,69.1,100,2026-01-10
2,S0001,C002,G03,C002-QZ,Quiz 3,quiz,84.6,100,2026-01-24
3,S0001,C002,G03,C002-QZ,Quiz 4,quiz,88.2,100,2026-02-07
4,S0001,C002,G03,C002-AS,Assignment 1,assignment,67.4,100,2026-02-21


## 1C · Load & Stack Excel Sheets (attendance)

In [ ]:
attendance_sheets = pd.read_excel(f'{DATA_PATH}attendance.xlsx', sheet_name=None)

print(f"\U0001f4c4 attendance.xlsx: {len(attendance_sheets)} sheets")
for sheet_name, df in attendance_sheets.items():
    print(f"\n   Sheet: '{sheet_name}' -> {df.shape[0]} rows x {df.shape[1]} cols")
    print(f"   Columns: {list(df.columns)}")
    if 'status' in df.columns:
        print(f"   Status values: {df['status'].unique()}")
    else:
        print(f"   \u26a0\ufe0f NO 'status' COLUMN — check column names")
    for col in df.columns:
        if 'session' in col.lower() and 'date' in col.lower():
            print(f"   DateTime column name: '{col}'")

📄 attendance.xlsx: 6 sheets

   Sheet: '2025-12' -> 2111 rows x 6 cols
   Columns: ['record_id', 'student_id', 'group_id', 'session_type', 'session_datetime', 'status']
   Status values: ['attended' 'absent' 'Atttended']
   DateTime column name: 'session_datetime'

   Sheet: '2026-01' -> 2341 rows x 6 cols
   Columns: ['record_id', 'student_id', 'group_id', 'session_type', 'session_datetime', 'status']
   Status values: ['Present' 'Absent']
   DateTime column name: 'session_datetime'

   Sheet: '2026-02' -> 2000 rows x 6 cols
   Columns: ['record_id', 'student_id', 'group_id', 'session_type', 'datetime', 'status']
   Status values: [1 0]

   Sheet: '2026-03' -> 2104 rows x 6 cols
   Columns: ['record_id', 'student_id', 'group_id', 'session_type', 'session_datetime', 'status']
   Status values: ['P' 'A']
   DateTime column name: 'session_datetime'

   Sheet: '2026-04' -> 2217 rows x 6 cols
   Columns: ['record_id', 'student_id', 'group_id', 'session_type', 'session_datetime', 'status']


### cleaning log Detected invalid negative scores.
Decision pending.

| # | Issue                    |
| - | ------------------------ |
| 1| Negative Score           |
| 2| Atttended typo           |
| 3| Present/Absent encoding  |
|4 | 1/0 encoding             |
|5 | datetime column mismatch |
|6 | P/A encoding             |
|7 | yes/no encoding          |
|8 | True/False encoding      |


In [ ]:


reconciled_sheets = []

for sheet_name, df in attendance_sheets.items():
    df = df.copy()
    df['source_sheet'] = sheet_name

    # FIX 1: Renamed columns
    col_mapping = {}
    for col in df.columns:
        if col != 'session_datetime' and 'session' in col.lower() and 'date' in col.lower():
            col_mapping[col] = 'session_datetime'
            p = next_problem()
            log_problem(p, f'attendance.xlsx ({sheet_name})',
                       f"Column named '{col}' instead of 'session_datetime'",
                       f"Renamed to 'session_datetime'")
    if col_mapping:
        df.rename(columns=col_mapping, inplace=True)

    # FIX 2: Status encoding
    original_statuses = df['status'].unique()
    status_map = {}
    for val in original_statuses:
        val_lower = str(val).strip().lower()
        if val_lower in ['attended', 'present', 'p', '1', '1.0', 'true', 'yes','atttended']:
            status_map[val] = 'attended'
        elif val_lower in ['absent', 'a', '0', '0.0', 'false', 'no']:
            status_map[val] = 'absent'
        else:
            status_map[val] = val
            print(f"   \u26a0\ufe0f Unknown status: '{val}' in '{sheet_name}'")

    if not all(v == k for k, v in status_map.items()):
        p = next_problem()
        log_problem(p, f'attendance.xlsx ({sheet_name})',
                   f"Status encoded as {list(set(original_statuses))} instead of attended/absent",
                   f"Mapped: {status_map}")

    df['status'] = df['status'].map(status_map)
    reconciled_sheets.append(df)

attendance = pd.concat(reconciled_sheets, ignore_index=True)
print(f"\n\u2705 Attendance reconciled: {attendance.shape[0]} rows x {attendance.shape[1]} cols")
print(f"   Final status values: {attendance['status'].unique()}")

🔴 Problem #1: Status encoded as ['Atttended', 'attended', 'absent'] instead of attended/absent
🔴 Problem #2: Status encoded as ['Absent', 'Present'] instead of attended/absent
🔴 Problem #3: Status encoded as [np.int64(0), np.int64(1)] instead of attended/absent
🔴 Problem #4: Status encoded as ['A', 'P'] instead of attended/absent
🔴 Problem #5: Status encoded as ['yes', 'no'] instead of attended/absent
🔴 Problem #6: Status encoded as [np.False_, np.True_] instead of attended/absent

✅ Attendance reconciled: 13010 rows x 8 cols
   Final status values: ['attended' 'absent']


In [ ]:
# FIX 1: Column name reconciliation
col_mapping = {}

for col in df.columns:
    col_lower = col.lower().strip()

    # Case 1: datetime -> session_datetime
    if col_lower == "datetime":
        col_mapping[col] = "session_datetime"

        p = next_problem()
        log_problem(
            p,
            f'attendance.xlsx ({sheet_name})',
            "Column named 'datetime' instead of 'session_datetime'",
            "Renamed to 'session_datetime'"
        )

    # Case 2: session_date -> session_datetime
    elif (
        col != "session_datetime"
        and "session" in col_lower
        and "date" in col_lower
    ):
        col_mapping[col] = "session_datetime"

        p = next_problem()
        log_problem(
            p,
            f'attendance.xlsx ({sheet_name})',
            f"Column named '{col}' instead of 'session_datetime'",
            "Renamed to 'session_datetime'"
        )

if col_mapping:
    df.rename(columns=col_mapping, inplace=True)

probelm: session_datetime and datetime were inconsistant in one month has nan and nat

In [ ]:
attendance[
    attendance['datetime'].notna()
][['source_sheet','session_datetime','datetime']].head()

,source_sheet,session_datetime,datetime
4452,2026-02,NaT,2026-02-05 16:00:00
4453,2026-02,NaT,2026-02-05 16:00:00
4454,2026-02,NaT,2026-02-05 16:00:00
4455,2026-02,NaT,2026-02-05 16:00:00
4456,2026-02,NaT,2026-02-05 16:00:00


In [ ]:
# Fill missing session_datetime values from datetime
attendance['session_datetime'] = attendance['session_datetime'].fillna(
    attendance['datetime']
)

# Verify
print("Missing session_datetime:",
      attendance['session_datetime'].isna().sum())

print("Rows with datetime values:",
      attendance['datetime'].notna().sum())

Missing session_datetime: 0
Rows with datetime values: 2000


In [ ]:
attendance['datetime'].notna().sum()

np.int64(2000)

In [ ]:
attendance[
    attendance['source_sheet'] == '2026-02'
][['session_datetime','datetime']].head()

,session_datetime,datetime
4452,2026-02-05 16:00:00,2026-02-05 16:00:00
4453,2026-02-05 16:00:00,2026-02-05 16:00:00
4454,2026-02-05 16:00:00,2026-02-05 16:00:00
4455,2026-02-05 16:00:00,2026-02-05 16:00:00
4456,2026-02-05 16:00:00,2026-02-05 16:00:00


In [ ]:
attendance.drop(columns=['datetime'], inplace=True)

In [ ]:
print(attendance.columns.tolist())

['record_id', 'student_id', 'group_id', 'session_type', 'session_datetime', 'status', 'source_sheet']


In [ ]:
attendance.head()

,record_id,student_id,group_id,session_type,session_datetime,status,source_sheet
0,AT000001,S0005,G01,session,2025-12-04 16:00:00,attended,2025-12
1,AT000002,S0009,G01,session,2025-12-04 16:00:00,attended,2025-12
2,AT000003,S0014,G01,session,2025-12-04 16:00:00,attended,2025-12
3,AT000004,S0016,G01,session,2025-12-04 16:00:00,attended,2025-12
4,AT000005,S0021,G01,session,2025-12-04 16:00:00,attended,2025-12


In [ ]:
attendance.to_csv(
    "cleaned_data/attendance_clean.csv",
    index=False
)

---
# ═══════════════════════════════════════════
# SECTION 2 · FIRST LOOK AT EVERY FILE
# ═══════════════════════════════════════════

> Snapshot every file before cleaning — know what you're dealing with.

In [ ]:
all_dfs = {
    'courses': courses,
    'groups': groups,
    'students': students,
    'grades': grades,
    'attendance': attendance,
    'concepts': concepts,
    'engagement': engagement,
    'submissions': submissions
}

for name, df in all_dfs.items():
    print(f"\n{'='*60}")
    print(f"\U0001f4ca {name.upper()}: {df.shape}")
    print(f"{'='*60}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNulls:\n{df.isnull().sum()}")
    print(f"\nDuplicates: {df.duplicated().sum()}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\nNumeric summary:\n{df[numeric_cols].describe()}")
    print()


📊 COURSES: (7, 8)

Dtypes:
course_id            object
course_name          object
category             object
difficulty_level     object
duration_weeks        int64
short_description    object
modules              object
is_active              bool
dtype: object

Nulls:
course_id            0
course_name          0
category             0
difficulty_level     0
duration_weeks       0
short_description    0
modules              0
is_active            0
dtype: int64

Duplicates: 0

Numeric summary:
       duration_weeks
count        7.000000
mean        12.571429
std          2.992053
min          8.000000
25%         11.000000
50%         12.000000
75%         15.000000
max         16.000000


📊 GROUPS: (12, 7)

Dtypes:
group_id               object
group_name             object
course_id              object
stated_num_students     int64
session_day            object
session_time           object
instructor             object
dtype: object

Nulls:
group_id               0
group_name  

---
# ═══════════════════════════════════════════
# SECTION 3 · SINGLE-FILE CLEANING
# ═══════════════════════════════════════════

> File by file: duplicates -> nulls -> impossible values -> inconsistent categoricals -> date/format issues.

## 3A · courses.csv

In [ ]:
print("=== COURSES ===")
print(f"Shape: {courses.shape}")
print(f"\nDuplicates: {courses.duplicated().sum()}")
print(f"Duplicate course_id: {courses['course_id'].duplicated().sum()}")
print(f"\nNulls:\n{courses.isnull().sum()}")
print(f"\ncategory values: {courses['category'].unique()}")
print(f"difficulty_level values: {courses['difficulty_level'].unique()}")
print(f"is_active values: {courses['is_active'].unique()}")
print(f"duration_weeks range: {courses['duration_weeks'].min()} - {courses['duration_weeks'].max()}")

=== COURSES ===
Shape: (7, 8)

Duplicates: 0
Duplicate course_id: 0

Nulls:
course_id            0
course_name          0
category             0
difficulty_level     0
duration_weeks       0
short_description    0
modules              0
is_active            0
dtype: int64

category values: ['Analytics' 'Programming' 'Design' 'Business' 'Security']
difficulty_level values: ['Beginner' 'Intermediate' 'Advanced']
is_active values: [ True]
duration_weeks range: 8 - 16


| Step | Action                                                |
| ---- | ----------------------------------------------------- |
| T001 | Parsed modules column from JSON string to Python list |


In [ ]:
# Courses — value checks

# Negative or zero duration
bad_duration = courses[courses['duration_weeks'] <= 0]
if len(bad_duration) > 0:
    p = next_problem()
    log_problem(p, 'courses.csv',
               f"{len(bad_duration)} courses with duration_weeks <= 0",
               "Flag for investigation")

# Valid difficulty levels
valid_difficulties = {'Beginner', 'Intermediate', 'Advanced'}
actual_difficulties = set(courses['difficulty_level'].dropna().unique())
unexpected = actual_difficulties - valid_difficulties
if unexpected:
    p = next_problem()
    log_problem(p, 'courses.csv',
               f"Unexpected difficulty levels: {unexpected}",
               "Standardize to Beginner/Intermediate/Advanced")

print(f"is_active dtype: {courses['is_active'].dtype}")
print(f"is_active unique: {courses['is_active'].unique()}")

is_active dtype: bool
is_active unique: [ True]


In [ ]:
bad_duration = courses[courses['duration_weeks'] <= 0]

In [ ]:
len(bad_duration) == 0

True

In [ ]:
courses.head()

,course_id,course_name,category,difficulty_level,duration_weeks,short_description,modules,is_active
0,C001,Data Analytics Fundamentals,Analytics,Beginner,12,Data Analytics Fundamentals — hands-on analytics track at Kayfa.,"[""Spreadsheets & Tabular Data"", ""Descriptive Statistics"", ""Data Cleaning"", ""...",True
1,C002,Python Programming,Programming,Beginner,14,Python Programming — hands-on programming track at Kayfa.,"[""Variables & Types"", ""Control Flow"", ""Functions"", ""Data Structures"", ""Recur...",True
2,C003,Web Development,Programming,Intermediate,16,Web Development — hands-on programming track at Kayfa.,"[""HTML & Semantics"", ""CSS Layout"", ""JavaScript DOM"", ""HTTP & APIs"", ""Respons...",True
3,C004,UI/UX Design,Design,Beginner,10,UI/UX Design — hands-on design track at Kayfa.,"[""Design Principles"", ""Wireframing"", ""Color Theory"", ""Prototyping"", ""Usabili...",True
4,C005,Digital Marketing,Business,Beginner,8,Digital Marketing — hands-on business track at Kayfa.,"[""SEO Basics"", ""Content Strategy"", ""Paid Ads"", ""Funnel Analytics""]",True


In [ ]:
courses.to_csv(
    "cleaned_data/courses.csv",
    index=False
)

## 3B · groups.csv

In [ ]:
print("=== GROUPS ===")
print(f"Shape: {groups.shape}")
print(f"\nDuplicates: {groups.duplicated().sum()}")
print(f"Duplicate group_id: {groups['group_id'].duplicated().sum()}")
print(f"\nNulls:\n{groups.isnull().sum()}")
print(f"\nsession_day values: {groups['session_day'].unique()}")
print(f"session_time samples: {groups['session_time'].unique()}")
print(f"stated_num_students range: {groups['stated_num_students'].min()} - {groups['stated_num_students'].max()}")
print(f"instructor values: {groups['instructor'].unique()}")

=== GROUPS ===
Shape: (12, 7)

Duplicates: 1
Duplicate group_id: 1

Nulls:
group_id               0
group_name             0
course_id              0
stated_num_students    0
session_day            0
session_time           0
instructor             1
dtype: int64

session_day values: ['Thursday' 'Sunday' 'Saturday' 'Tuesday' 'Wednesday' 'Friday']
session_time samples: ['16:00' '18:00' '6 PM' '1800' '17:00' '21:00' '19:00' '00:00']
stated_num_students range: 0 - 76
instructor values: ['Eng. Khaled Adel' 'Dr. Mona Saad' 'Dr. Laila ElBaz' 'Eng. Hossam Refaat'
 nan]


| Problem # | File       | Issue                              |
| --------- | ---------- | ---------------------------------- |
|  9| groups.csv | Duplicate row exists               |
| 10 | groups.csv | Duplicate group_id exists          |
|   11   | groups.csv | Missing instructor value           |
|    12  | groups.csv | Group with stated_num_students = 0 |
13 session_time values: inconsistant

In [ ]:
groups[groups.duplicated(keep=False)]

,group_id,group_name,course_id,stated_num_students,session_day,session_time,instructor
0,G01,Group 01 — C001,C001,52,Thursday,16:00,Eng. Khaled Adel
11,G01,Group 01 — C001,C001,52,Thursday,16:00,Eng. Khaled Adel


In [ ]:
groups = groups.drop_duplicates()

In [ ]:
# Groups — FK and consistency checks

# course_id FK integrity
invalid_course_refs = groups[~groups['course_id'].isin(courses['course_id'])]
if len(invalid_course_refs) > 0:
    p = next_problem()
    log_problem(p, 'groups.csv -> courses.csv',
               f"{len(invalid_course_refs)} groups reference non-existent course_ids: {invalid_course_refs['course_id'].tolist()}",
               "Flag — broken FK")

# session_time format consistency
print("session_time values:")
for t in groups['session_time'].unique():
    print(f"  '{t}'")

# session_day typos
valid_days = {'Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'}
actual_days = set(groups['session_day'].dropna().str.strip().unique())
unexpected_days = actual_days - valid_days
if unexpected_days:
    p = next_problem()
    log_problem(p, 'groups.csv',
               f"Unexpected session_day values: {unexpected_days}",
               "Standardize spelling")

# Whitespace in text columns
for col in ['group_name', 'instructor', 'session_day']:
    if col in groups.columns:
        stripped = groups[col].astype(str).str.strip()
        has_ws = (groups[col].astype(str) != stripped).sum()
        if has_ws > 0:
            p = next_problem()
            log_problem(p, 'groups.csv',
                       f"{has_ws} rows in '{col}' have leading/trailing whitespace",
                       "Stripped whitespace")
            groups[col] = stripped

session_time values:
  '16:00'
  '18:00'
  '6 PM'
  '1800'
  '17:00'
  '21:00'
  '19:00'
  '00:00'


In [ ]:
groups['session_time'].value_counts(dropna=False)

,count
session_time,
16:00,2
18:00,2
17:00,2
6 PM,1
1800,1
21:00,1
19:00,1
00:00,1


In [ ]:
groups[groups['session_time'] == '00:00']

,group_id,group_name,course_id,stated_num_students,session_day,session_time,instructor


In [ ]:
groups = groups[
    groups['group_id'] != 'G99'
]

In [ ]:
from datetime import datetime

def normalize_time(value):
    value = str(value).strip()

    formats = [
        "%H:%M",   # 16:00
        "%I %p",   # 6 PM
        "%H%M"     # 1800
    ]

    for fmt in formats:
        try:
            return datetime.strptime(
                value,
                fmt
            ).strftime("%H:%M")
        except:
            pass

    return value

groups['session_time'] = groups['session_time'].apply(
    normalize_time
)

In [ ]:
print(
    sorted(
        groups['session_time'].unique()
    )
)

['16:00', '17:00', '18:00', '19:00', '21:00']


In [ ]:
groups['session_time'] = pd.to_datetime(
    groups['session_time'],
    format='%H:%M'
).dt.time

| Problem # | Issue                                | Status  |
| --------- | ------------------------------------ | ------- |
| 14   | Test group record (G99)              | ✅ Fixed |
| 15    | Missing instructor (inside G99)      | ✅ Fixed |
| 16     | stated_num_students = 0 (inside G99) | ✅ Fixed |
| 17      | Suspicious time 00:00 (inside G99)   | ✅ Fixed |


In [ ]:
groups.head()

,group_id,group_name,course_id,stated_num_students,session_day,session_time,instructor
0,G01,Group 01 — C001,C001,52,Thursday,16:00:00,Eng. Khaled Adel
1,G02,Group 02 — C001,C001,56,Thursday,18:00:00,Dr. Mona Saad
2,G03,Group 03 — C002,C002,67,Sunday,18:00:00,Dr. Laila ElBaz
3,G04,Group 04 — C002,C002,65,Saturday,16:00:00,Eng. Hossam Refaat
4,G05,Group 05 — C003,C003,76,Tuesday,18:00:00,Eng. Khaled Adel


In [ ]:
groups.duplicated().sum()

np.int64(0)

In [ ]:
groups.isna().sum()

,0
group_id,0
group_name,0
course_id,0
stated_num_students,0
session_day,0
session_time,0
instructor,0


In [ ]:
groups.to_csv(
    "cleaned_data/groups_clean.csv",
    index=False
)

## 3C · students.csv

In [ ]:
print("=== STUDENTS ===")
print(f"Shape: {students.shape}")
print(f"\nDuplicates: {students.duplicated().sum()}")
print(f"Duplicate student_id: {students['student_id'].duplicated().sum()}")
print(f"\nNulls:\n{students.isnull().sum()}")
print(f"\ngender values: {students['gender'].unique()}")
print(f"city values ({students['city'].nunique()} unique): {students['city'].unique()}")
print(f"age range: {students['age'].min()} - {students['age'].max()}")
print(f"\nAge distribution:")
print(students['age'].describe())

=== STUDENTS ===
Shape: (502, 8)

Duplicates: 0
Duplicate student_id: 2

Nulls:
student_id         0
full_name          4
age                0
gender             0
city               0
email              0
group_id           0
enrollment_date    0
dtype: int64

gender values: ['Male' 'Female' 'MALE' 'F' 'm' 'f' 'M' 'Fem' 'male' 'female']
city values (10 unique): ['Mansoura' 'Zagazig' 'Ismailia' 'Giza' 'Asyut' 'Alexandria' 'Port Said'
 'Tanta' 'Fayoum' 'Cairo']
age range: -22 - 121

Age distribution:
count    502.000000
mean      21.750996
std        7.442645
min      -22.000000
25%       19.000000
50%       21.000000
75%       23.000000
max      121.000000
Name: age, dtype: float64


| Problem # | Issue                                      | Status         |
| --------- | ------------------------------------------ | -------------- |
|     18 | Duplicate student_id values (2 duplicates) | ⚠️ Investigate |
|     19  | Missing full_name values (4 records)       | ⚠️ Detected    |
|   20  | Invalid negative age (-22)                 | ⚠️ Detected    |
|   21 | Unrealistic age (121)                      | ⚠️ Detected    |
|  22  | Inconsistent gender encoding               | ⚠️ Detected    |
| 23   | duplicate mails              | ⚠️ Detected    |
|  24  | duplicate names with diffrent features            | ⚠️ Detected    |

In [ ]:
students[
    students['student_id'].duplicated(
        keep=False
    )
].sort_values('student_id')

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
73,S0074,Sherif Zaki,25,Female,Alexandria,sherif.zaki@kayfa-student.io,G08,2025-12-09
501,S0074,Sherif Zaki,88,Female,Alexandria,sherif.zaki@kayfa-student.io,G08,2025-12-09
361,S0362,Hana Mansour,17,Male,Tanta,hana.mansour@kayfa-student.io,G08,2025-12-16
500,S0362,Hana Mansour,99,Male,Tanta,hana.mansour@kayfa-student.io,G08,2025-12-16


### Duplicate Records with corrupted age values.

In [ ]:
students = students[
    ~(
        ((students['student_id'] == 'S0074') & (students['age'] == 88))
        |
        ((students['student_id'] == 'S0362') & (students['age'] == 99))
    )
].copy()

In [ ]:
print(students['student_id'].duplicated().sum())

0


In [ ]:
students['age'].value_counts().sort_index()

,count
age,
-22,1
-5,1
4,1
17,65
18,28
19,49
20,53
21,66
22,60


In [ ]:


# AGE — impossible values
bad_age = students[(students['age'] < 16) | (students['age'] > 65)]
if len(bad_age) > 0:
    p = next_problem()
    log_problem(p, 'students.csv',
               f"{len(bad_age)} students with suspicious age: {bad_age['age'].tolist()}",
               "Flag / impute with group median")
    print(bad_age[['student_id', 'full_name', 'age']])

# GENDER — inconsistent encoding
print(f"\nGender unique (raw): {students['gender'].unique()}")
gender_normalized = students['gender'].str.strip().str.title()
if not (students['gender'] == gender_normalized).all():
    p = next_problem()
    log_problem(p, 'students.csv',
               "Gender has inconsistent casing/whitespace",
               "Normalized to Title Case")
    students['gender'] = gender_normalized



🔴 Problem #7: 4 students with suspicious age: [4, -22, -5, 121]
    student_id       full_name  age
22       S0023     Tarek Gamal    4
208      S0209   Marwan Naguib  -22
452      S0453    Marwan ElBaz   -5
477      S0478  Nada Abdelaziz  121

Gender unique (raw): ['Male' 'Female' 'MALE' 'F' 'm' 'f' 'M' 'Fem' 'male' 'female']
🔴 Problem #8: Gender has inconsistent casing/whitespace


In [ ]:
# EMAIL — format validation
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
invalid_emails = students[~students['email'].astype(str).str.match(email_pattern)]
if len(invalid_emails) > 0:
    p = next_problem()
    log_problem(p, 'students.csv',
               f"{len(invalid_emails)} invalid email formats",
               "Flagged for review")
    print(invalid_emails[['student_id', 'email']])

# DUPLICATE EMAILS
dup_emails = students[students['email'].duplicated(keep=False)]
if len(dup_emails) > 0:
    p = next_problem()
    log_problem(p, 'students.csv',
               f"{len(dup_emails)} rows share duplicate email addresses",
               "Flagged — possible data entry error")
    print(dup_emails[['student_id', 'full_name', 'email']])

🔴 Problem #9: 4 invalid email formats
   student_id             email
14      S0015          missing@
22      S0023         @kayfa.io
60      S0061  spaces here@x.io
61      S0062      not-an-email
🔴 Problem #10: 173 rows share duplicate email addresses
    student_id        full_name                             email
0        S0001       Hana Gamal       hana.gamal@kayfa-student.io
4        S0005   Habiba Mahmoud   habiba.mahmoud@kayfa-student.io
5        S0006      Aya ElSayed      aya.elsayed@kayfa-student.io
7        S0008       Mona Gamal       mona.gamal@kayfa-student.io
10       S0011       Menna Saad       menna.saad@kayfa-student.io
..         ...              ...                               ...
491      S0492  Youssef Ibrahim  youssef.ibrahim@kayfa-student.io
492      S0493    Tarek ElMasry    tarek.elmasry@kayfa-student.io
493      S0494    Hassan Refaat    hassan.refaat@kayfa-student.io
495      S0496    Farida Sultan    farida.sultan@kayfa-student.io
499      S0500  Adel

| Problem # | Issue                                                                                                         | Status       |
| --------- | ------------------------------------------------------------------------------------------------------------- | ------------ |
|25      | Non-unique email addresses shared across multiple student identities (78 unique emails affecting 173 records) | ⚠️ Confirmed |


In [ ]:
dup_emails = students[
    students['email'].duplicated(keep=False)
]

print("Rows involved:", len(dup_emails))

print(
    "Unique duplicate emails:",
    dup_emails['email'].nunique()
)

Rows involved: 173
Unique duplicate emails: 78


In [ ]:
dup_emails[
    dup_emails['email'] ==
    dup_emails['email'].value_counts().index[0]
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
229,S0230,Dina Zaki,26,Male,Ismailia,dina.zaki@kayfa-student.io,G04,2025-12-09
261,S0262,Dina Zaki,22,Female,Mansoura,dina.zaki@kayfa-student.io,G01,2025-12-09
319,S0320,Dina Zaki,21,Female,Mansoura,dina.zaki@kayfa-student.io,G08,2025-12-01
327,S0328,Dina Zaki,24,Male,Cairo,dina.zaki@kayfa-student.io,G04,2025-12-13
378,S0379,Dina Zaki,17,Male,Mansoura,dina.zaki@kayfa-student.io,G03,2025-12-20


In [ ]:
students.groupby('email').agg({
    'student_id':'count',
    'full_name':'nunique'
}).sort_values('student_id', ascending=False).head(20)

,student_id,full_name
email,,
dina.zaki@kayfa-student.io,5,1
hana.awad@kayfa-student.io,4,1
nada.abdelhamid@kayfa-student.io,3,1
salma.elmasry@kayfa-student.io,3,1
marwan.elshafei@kayfa-student.io,3,1
hana.mansour@kayfa-student.io,3,1
omar.elmasry@kayfa-student.io,3,1
hassan.nasr@kayfa-student.io,3,1
mohamed.mahmoud@kayfa-student.io,3,1


Identity analysis revealed that 78 email addresses are reused across 173 student records. Unlike simple email duplication, the duplicated records also share the same full name while being assigned different student IDs. This suggests that the dataset contains multiple identities representing the same individual, indicating a student identity duplication issue rather than isolated email-entry errors.

In [ ]:
students[
    students['email'] == 'dina.zaki@kayfa-student.io'
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
229,S0230,Dina Zaki,26,Male,Ismailia,dina.zaki@kayfa-student.io,G04,2025-12-09
261,S0262,Dina Zaki,22,Female,Mansoura,dina.zaki@kayfa-student.io,G01,2025-12-09
319,S0320,Dina Zaki,21,Female,Mansoura,dina.zaki@kayfa-student.io,G08,2025-12-01
327,S0328,Dina Zaki,24,Male,Cairo,dina.zaki@kayfa-student.io,G04,2025-12-13
378,S0379,Dina Zaki,17,Male,Mansoura,dina.zaki@kayfa-student.io,G03,2025-12-20


In [ ]:
dup_emails = (
    students
    .groupby('email')
    .agg({
        'student_id':'nunique',
        'full_name':'nunique',
        'gender':'nunique',
        'city':'nunique',
        'group_id':'nunique'
    })
    .sort_values(
        'student_id',
        ascending=False
    )
)

dup_emails.head(20)

,student_id,full_name,gender,city,group_id
email,,,,,
dina.zaki@kayfa-student.io,5,1,2,3,4
hana.awad@kayfa-student.io,4,1,2,4,3
sara.darwish@kayfa-student.io,3,1,1,3,3
mohamed.mahmoud@kayfa-student.io,3,1,2,3,3
ziad.gamal@kayfa-student.io,3,1,2,3,3
malak.abdelaziz@kayfa-student.io,3,1,1,3,3
salma.elmasry@kayfa-student.io,3,1,1,3,3
nada.abdelhamid@kayfa-student.io,3,1,2,3,3
mona.gamal@kayfa-student.io,3,1,2,2,3


In [ ]:
email = 'dina.zaki@kayfa-student.io'

students[
    students['email'] == email
][['student_id','full_name','group_id']]

,student_id,full_name,group_id
229,S0230,Dina Zaki,G04
261,S0262,Dina Zaki,G01
319,S0320,Dina Zaki,G08
327,S0328,Dina Zaki,G04
378,S0379,Dina Zaki,G03


In [ ]:
ids = ['S0230','S0262','S0320','S0328','S0379']

for sid in ids:
    print(sid)

    print("Grades:",
          grades['student_id'].eq(sid).sum())

    print("Attendance:",
          attendance['student_id'].eq(sid).sum())

    print("Engagement:",
          engagement['student_id'].eq(sid).sum())

    print("Submissions:",
          submissions['student_id'].eq(sid).sum())

    print("-"*40)

S0230
Grades: 11
Attendance: 26
Engagement: 67
Submissions: 3
----------------------------------------
S0262
Grades: 11
Attendance: 26
Engagement: 72
Submissions: 3
----------------------------------------
S0320
Grades: 11
Attendance: 26
Engagement: 88
Submissions: 3
----------------------------------------
S0328
Grades: 11
Attendance: 26
Engagement: 95
Submissions: 3
----------------------------------------
S0379
Grades: 11
Attendance: 26
Engagement: 52
Submissions: 3
----------------------------------------


In [ ]:
ids = ['S0230','S0262','S0320','S0328','S0379']

grades[
    grades['student_id'].isin(ids)
].groupby('student_id')['score'].mean()

,score
student_id,
S0230,75.354545
S0262,69.018182
S0320,71.436364
S0328,69.272727
S0379,70.972727


In [ ]:
engagement[
    engagement['student_id'].isin(ids)
].groupby('student_id')['duration_seconds'].sum()

,duration_seconds
student_id,
S0230,13533.0
S0262,15482.0
S0320,17723.0
S0328,18755.0
S0379,8020.0


In [ ]:
ids = ['S0230','S0262','S0320','S0328','S0379']

students[
    students['student_id'].isin(ids)
][[
    'student_id',
    'full_name',
    'email',
    'city',
    'group_id'
]]

,student_id,full_name,email,city,group_id
229,S0230,Dina Zaki,dina.zaki@kayfa-student.io,Ismailia,G04
261,S0262,Dina Zaki,dina.zaki@kayfa-student.io,Mansoura,G01
319,S0320,Dina Zaki,dina.zaki@kayfa-student.io,Mansoura,G08
327,S0328,Dina Zaki,dina.zaki@kayfa-student.io,Cairo,G04
378,S0379,Dina Zaki,dina.zaki@kayfa-student.io,Mansoura,G03


In [ ]:
students.groupby('full_name').size().sort_values(ascending=False).head(20)

,0
full_name,
Dina Zaki,5
Hana Awad,4
Farida Darwish,3
Mohamed Mahmoud,3
Ziad Gamal,3
Sara Darwish,3
Salma ElMasry,3
Nada AbdelHamid,3
Malak Abdelaziz,3


Student names are not unique within the dataset.
Email addresses were generated solely from first and last names,
which caused collisions whenever multiple students shared the same name.

In [ ]:
students[
    students['email'] == 'hana.awad@kayfa-student.io'
][[
    'student_id',
    'full_name',
    'age',
    'gender',
    'city',
    'group_id'
]]

,student_id,full_name,age,gender,city,group_id
31,S0032,Hana Awad,19.0,Male,Mansoura,G01
263,S0264,Hana Awad,17.0,Male,Alexandria,G02
328,S0329,Hana Awad,22.0,Female,Asyut,G01
357,S0358,Hana Awad,17.0,Female,Port Said,G03


In [ ]:
ids = ['S0032','S0264','S0329','S0358']

for sid in ids:
    print(sid)

    print("Grades:",
          grades['student_id'].eq(sid).sum())

    print("Attendance:",
          attendance['student_id'].eq(sid).sum())

    print("Engagement:",
          engagement['student_id'].eq(sid).sum())

    print("Submissions:",
          submissions['student_id'].eq(sid).sum())

    print("-"*40)

S0032
Grades: 11
Attendance: 26
Engagement: 50
Submissions: 3
----------------------------------------
S0264
Grades: 11
Attendance: 26
Engagement: 88
Submissions: 3
----------------------------------------
S0329
Grades: 11
Attendance: 26
Engagement: 51
Submissions: 3
----------------------------------------
S0358
Grades: 11
Attendance: 26
Engagement: 50
Submissions: 3
----------------------------------------


In [ ]:
# Find duplicated emails
dup_mask = students.duplicated(
    subset=['email'],
    keep=False
)

# Process duplicated emails only
for email in students.loc[dup_mask, 'email'].unique():

    idx = students[
        students['email'] == email
    ].index

    # Keep first email unchanged
    first = True

    for i in idx:

        if first:
            first = False
            continue

        username, domain = students.loc[i, 'email'].split('@')

        sid = students.loc[i, 'student_id'].lower()

        students.loc[i, 'email'] = (
            f"{username}.{sid}@{domain}"
        )

# Verify
print(
    "Remaining duplicate emails:",
    students['email'].duplicated().sum()
)

Remaining duplicate emails: 0


In [ ]:
students[students['full_name'].isna()]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
108,S0109,NaN,22,Female,Mansoura,khaled.gamal@kayfa-student.io,G04,2025-12-17
134,S0135,NaN,25,Female,Tanta,farida.elbaz@kayfa-student.io,G02,2025-12-10
286,S0287,NaN,20,Male,Giza,nada.gamal@kayfa-student.io,G06,2025-12-13
365,S0366,NaN,20,Female,Asyut,fady.elghandour@kayfa-student.io,G03,2025-12-10


### null in names column

In [ ]:
def name_from_email(email):
    username = email.split('@')[0]

    parts = username.split('.')

    return ' '.join(
        word.capitalize()
        for word in parts
    )

students.loc[
    students['full_name'].isna(),
    'full_name'
] = students.loc[
    students['full_name'].isna(),
    'email'
].apply(name_from_email)

In [ ]:
students['gender'].value_counts()

,count
gender,
Female,260
Male,235
F,2
M,2
Fem,1


In [ ]:
gender_map = {
    'M': 'Male',
    'm': 'Male',
    'MALE': 'Male',
    'male': 'Male',
    'Male': 'Male',

    'F': 'Female',
    'f': 'Female',
    'Fem': 'Female',
    'female': 'Female',
    'Female': 'Female'
}

students['gender'] = students['gender'].map(gender_map)

| Problem # | Issue                                                            | Insight                                                                                                                             | Fix                                                                    |
| --------- | ---------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------- |
| 26     | Inconsistent gender encoding (`F`, `M`, `Fem`, `Male`, `Female`) | Gender values were stored using multiple representations for the same category, creating inconsistencies in analysis and reporting. | Standardized all gender values to two categories: `Male` and `Female`. |


In [ ]:
students[
    students['age'].isin([-22, -5, 4, 121])
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
22,S0023,Tarek Gamal,4,Female,Mansoura,@kayfa.io,G08,2025-12-19
208,S0209,Marwan Naguib,-22,Female,Giza,marwan.naguib@kayfa-student.io,G03,2025-12-12
452,S0453,Marwan ElBaz,-5,Male,Zagazig,marwan.elbaz@kayfa-student.io,G07,2025-12-10
477,S0478,Nada Abdelaziz,121,Male,Tanta,nada.abdelaziz@kayfa-student.io,G09,2025-12-05


In [ ]:
suspects = ['S0023', 'S0209', 'S0453', 'S0478']

In [ ]:
grades[
    grades['student_id'].isin(suspects)
].head(20)

,student_id,course_id,group_id,assessment_id,assessment_title,type,score,max_score,date
244,S0023,C006,G08,C006-QZ,Quiz 1,quiz,70.2,100,2026-01-01
245,S0023,C006,G08,C006-QZ,Quiz 2,quiz,75.0,100,2026-01-15
246,S0023,C006,G08,C006-QZ,Quiz 3,quiz,73.2,100,2026-01-28
247,S0023,C006,G08,C006-QZ,Quiz 4,quiz,88.0,100,2026-02-11
248,S0023,C006,G08,C006-AS,Assignment 1,assignment,92.2,100,2026-02-24
249,S0023,C006,G08,C006-AS,Assignment 2,assignment,51.8,100,2026-03-10
250,S0023,C006,G08,C006-AS,Assignment 3,assignment,91.3,100,2026-03-23
251,S0023,C006,G08,C006-PR,Practical 1,practical,57.3,100,2026-04-06
252,S0023,C006,G08,C006-PR,Practical 2,practical,67.3,100,2026-04-19
253,S0023,C006,G08,C006-EX,Midterm Exam,exam,61.1,100,2026-05-03


In [ ]:
attendance[
    attendance['student_id'].isin(suspects)
][['student_id','group_id']].drop_duplicates()

,student_id,group_id
447,S0209,G03
1416,S0453,G07
1657,S0023,G08
1946,S0478,G09


In [ ]:
submissions[
    submissions['student_id'].isin(suspects)
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
66,SUB00067,S0023,C006,C006-AS,2026-02-24T23:59,2026-02-24T04:29,False,128,1
67,SUB00068,S0023,C006,C006-AS,2026-03-10T23:59,2026-03-11T03:32,True,75,1
68,SUB00069,S0023,C006,C006-AS,2026-03-23T23:59,2026-03-24T09:52,True,127,1
624,SUB00625,S0209,C002,C002-AS,2026-02-20T23:59,2026-02-21T00:52,True,120,1
625,SUB00626,S0209,C002,C002-AS,2026-03-06T23:59,2026-03-06T20:53,False,80,1
626,SUB00627,S0209,C002,C002-AS,2026-03-20T23:59,2026-03-20T10:56,False,131,1
1356,SUB01357,S0453,C005,C005-AS,2026-02-19T23:59,2026-02-20T05:12,True,103,3
1357,SUB01358,S0453,C005,C005-AS,2026-03-05T23:59,2026-03-05T19:22,False,110,3
1358,SUB01359,S0453,C005,C005-AS,2026-03-19T23:59,2026-03-20T03:27,True,115,1
1431,SUB01432,S0478,C006,C006-AS,2026-02-16T23:59,2026-02-18T14:02,True,146,1


In [ ]:
engagement[
    engagement['student_id'].isin(suspects)
]

,event_id,student_id,event_type,event_datetime,duration_seconds,device
1416,EV001417,S0023,video_watch,2026-03-17T18:30:52,312.0,web
1417,EV001418,S0023,forum_post,2026-05-17T02:47:21,NaN,web
1418,EV001419,S0023,resource_download,2026-03-28T09:41:55,NaN,web
1419,EV001420,S0023,video_watch,2026-05-16T22:57:47,445.0,mobile
1420,EV001421,S0023,login,2025-12-25T19:13:04,NaN,web
...,...,...,...,...,...,...
29559,EV029560,S0478,login,2026-05-09T07:52:05,NaN,web
29560,EV029561,S0478,login,2026-04-21T00:19:15,NaN,mobile
29561,EV029562,S0478,login,2025-12-05T12:42:17,NaN,web
29562,EV029563,S0478,login,2026-02-06T04:16:57,NaN,web


In [ ]:
concepts[
    concepts['student_id'].isin(suspects)
]

,record_id,student_id,course_id,assessment_id,question_no,concept_id,concept_name,score_pct,mastery_status,timestamp
528,CP000529,S0023,C006,C006-QZ,1,C006-K01,Regression,60.0,passed,2026-01-01T22:29:14
529,CP000530,S0023,C006,C006-QZ,2,C006-K06,Clustering,66.7,passed,2026-01-01T22:29:14
530,CP000531,S0023,C006,C006-QZ,3,C006-K03,Overfitting & Regularization,40.0,failed,2026-01-01T22:29:14
531,CP000532,S0023,C006,C006-QZ,4,C006-K05,Model Evaluation,60.2,passed,2026-01-01T22:29:14
532,CP000533,S0023,C006,C006-QZ,1,C006-K03,Overfitting & Regularization,61.8,passed,2026-01-15T22:29:14
533,CP000534,S0023,C006,C006-QZ,2,C006-K06,Clustering,77.5,passed,2026-01-15T22:29:14
534,CP000535,S0023,C006,C006-QZ,3,C006-K01,Regression,75.0,passed,2026-01-15T22:29:14
535,CP000536,S0023,C006,C006-QZ,4,C006-K04,Feature Engineering,69.5,passed,2026-01-15T22:29:14
536,CP000537,S0023,C006,C006-QZ,1,C006-K03,Overfitting & Regularization,54.7,failed,2026-01-28T22:29:14
537,CP000538,S0023,C006,C006-QZ,2,C006-K02,Classification,60.2,passed,2026-01-28T22:29:14


Cross-dataset validation showed that the affected students have valid activity records, grades, attendance, and submissions. Therefore, the issue is isolated to the demographic attributes in students.csv, rather than representing invalid or orphaned student records.

In [ ]:
students.loc[
    (students['age'] < 15) |
    (students['age'] > 100),
    'age'
] = np.nan

In [ ]:
students[
    students['email'].isin([
        'marwan.elbaz@kayfa-student.io',
        'nada.abdelaziz@kayfa-student.io'
    ])
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
452,S0453,Marwan ElBaz,NaN,Male,Zagazig,marwan.elbaz@kayfa-student.io,G07,2025-12-10
477,S0478,Nada Abdelaziz,NaN,Male,Tanta,nada.abdelaziz@kayfa-student.io,G09,2025-12-05


In [ ]:
students[
    students['email'] == 'marwan.naguib@kayfa-student.io'
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
208,S0209,Marwan Naguib,NaN,Female,Giza,marwan.naguib@kayfa-student.io,G03,2025-12-12
239,S0240,Marwan Naguib,21.0,Male,Fayoum,marwan.naguib@kayfa-student.io,G09,2025-12-08


In [ ]:
students[
    students['full_name'].isin([
        'Tarek Gamal',
        'Marwan Naguib',
        'Marwan ElBaz',
        'Nada Abdelaziz'
    ])
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
22,S0023,Tarek Gamal,NaN,Female,Mansoura,@kayfa.io,G08,2025-12-19
208,S0209,Marwan Naguib,NaN,Female,Giza,marwan.naguib@kayfa-student.io,G03,2025-12-12
239,S0240,Marwan Naguib,21.0,Male,Fayoum,marwan.naguib@kayfa-student.io,G09,2025-12-08
452,S0453,Marwan ElBaz,NaN,Male,Zagazig,marwan.elbaz@kayfa-student.io,G07,2025-12-10
477,S0478,Nada Abdelaziz,NaN,Male,Tanta,nada.abdelaziz@kayfa-student.io,G09,2025-12-05


In [ ]:
students.loc[
    students['student_id'] == 'S0023',
    'email'
] = 'tarek.gamal@kayfa-student.io'

In [ ]:
students.loc[
    students['student_id'] == 'S0209',
    'gender'
] = 'Male'

| Problem # | Issue                                             | Status      |
| --------- | ------------------------------------------------- | ----------- |
| 27      | Students reference non-existent groups (GZZ, G77) | ⚠️ Detected |


In [ ]:
students[
    students['group_id'].isin(['GZZ','G77'])
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
13,S0014,Ali Refaat,17.0,Male,Ismailia,ali.refaat@kayfa-student.io,GZZ,2025-12-18
94,S0095,Yasmin Fathy,22.0,Female,Zagazig,yasmin.fathy@kayfa-student.io,G77,2025-12-06
322,S0323,Aya Kamel,22.0,Female,Zagazig,aya.kamel@kayfa-student.io,G77,2025-12-18


looks like its a typo lets detect

In [ ]:
sorted(groups['group_id'].unique())

['G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10']

In [ ]:
attendance[
    attendance['student_id'].isin(
        ['S0014','S0095','S0323']
    )
][['student_id','group_id']].drop_duplicates()

,student_id,group_id
2,S0014,G01
11,S0095,G01
461,S0323,G03


In [ ]:
print("="*60)
print("FINAL STUDENTS VALIDATION")
print("="*60)

# Shape
print(f"\nRows: {len(students)}")

# Student ID
print("\n1. STUDENT IDs")
print("Duplicate student_ids:",
      students['student_id'].duplicated().sum())

# Full name
print("\n2. FULL NAMES")
print("Missing names:",
      students['full_name'].isna().sum())

# Age
print("\n3. AGE")
print("Missing ages:",
      students['age'].isna().sum())

print("Invalid ages (<15 or >60):",
      ((students['age'] < 15) |
       (students['age'] > 60)).sum())

# Gender
print("\n4. GENDER")
print("Unique values:",
      students['gender'].unique())

# Email
print("\n5. EMAIL")

invalid_email = ~students['email'].str.match(
    r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$',
    na=False
)

print("Invalid emails:",
      invalid_email.sum())

print("Duplicate emails:",
      students['email'].duplicated().sum())

# Group IDs
print("\n6. GROUP REFERENCES")

valid_groups = set(groups['group_id'])

invalid_groups = students[
    ~students['group_id'].isin(valid_groups)
]

print("Invalid group_ids:",
      len(invalid_groups))

# Enrollment date
print("\n7. ENROLLMENT DATE")

dates = pd.to_datetime(
    students['enrollment_date'],
    errors='coerce'
)

print("Invalid dates:",
      dates.isna().sum())

# Entire row duplicates
print("\n8. ROW DUPLICATES")
print("Duplicate rows:",
      students.duplicated().sum())

print("\n" + "="*60)
print("VALIDATION COMPLETE")
print("="*60)

FINAL STUDENTS VALIDATION

Rows: 500

1. STUDENT IDs
Duplicate student_ids: 0

2. FULL NAMES
Missing names: 0

3. AGE
Missing ages: 4
Invalid ages (<15 or >60): 0

4. GENDER
Unique values: ['Male' 'Female']

5. EMAIL
Invalid emails: 0
Duplicate emails: 0

6. GROUP REFERENCES
Invalid group_ids: 0

7. ENROLLMENT DATE
Invalid dates: 0

8. ROW DUPLICATES
Duplicate rows: 0

VALIDATION COMPLETE


In [ ]:
students[
    ~students['email'].str.match(
        r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$',
        na=False
    )
]

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
14,S0015,Fady ElMasry,25.0,Female,Asyut,missing@,G02,2025-12-08
60,S0061,Karim AbdelHamid,18.0,Male,Fayoum,spaces here@x.io,G03,2025-12-16
61,S0062,Menna ElGhandour,22.0,Female,Mansoura,not-an-email,G04,2025-12-04


In [ ]:
new_emails = [
    'fady.elmasry@kayfa-student.io',
    'karim.abdelhamid@kayfa-student.io',
    'menna.elghandour@kayfa-student.io'
]

students[
    students['email'].isin(new_emails)
][['student_id','full_name','email']]

,student_id,full_name,email


In [ ]:
students.loc[
    students['student_id'] == 'S0015',
    'email'
] = 'fady.elmasry@kayfa-student.io'

students.loc[
    students['student_id'] == 'S0061',
    'email'
] = 'karim.abdelhamid@kayfa-student.io'

students.loc[
    students['student_id'] == 'S0062',
    'email'
] = 'menna.elghandour@kayfa-student.io'

In [ ]:
grades[
    grades['student_id'].isin(
        ['S0014','S0095','S0323']
    )
][['student_id','group_id']].drop_duplicates()

,student_id,group_id
145,S0014,G01
1036,S0095,G01
3544,S0323,G03


In [ ]:
students = students.copy()

students.loc[
    students['student_id'] == 'S0014',
    'group_id'
] = 'G01'

students.loc[
    students['student_id'] == 'S0095',
    'group_id'
] = 'G01'

students.loc[
    students['student_id'] == 'S0323',
    'group_id'
] = 'G03'

| Problem # | Issue                         | Evidence / Insight                                                                                        | Resolution                                                                           |
| --------- | ----------------------------- | --------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------ |
|    28 | Duplicate Student IDs         | Two student IDs appeared more than once (`S0074`, `S0362`) with conflicting ages                          | Investigated and retained the most reliable records                                  |
|  29  | Missing Full Names            | 4 records had null values in `full_name`                                                                  | Reconstructed names from email usernames                                             |
| 30    | Invalid Age Values            | Detected implausible ages: `-22`, `-5`, `4`, and `121`                                                    | Replaced with `NaN` because no reliable source existed to infer the correct ages     |
|  31 | Inconsistent Gender Encoding  | Values included `Male`, `Female`, `M`, `F`, and `Fem`                                                     | Standardized all values to `Male` and `Female`                                       |
|  32  | Invalid Email Formats         | Examples included `missing@`, `@kayfa.io`, `spaces here@x.io`, and `not-an-email`                         | Corrected where reliable evidence existed; otherwise flagged for review              |
|    33  | Invalid Group References      | Students referenced non-existent groups (`GZZ`, `G77`)                                                    | Corrected using attendance and grades cross-validation                               |
|  34   | Duplicate Email Addresses     | 78 unique emails were shared across 173 student records                                                   | Investigated as identity-quality issues                                              |
|    35 | Identity Conflicts            | Same name and email appeared under multiple student IDs with conflicting attributes (e.g., Marwan Naguib) | Resolved attributes only when supported by evidence from duplicate records           |
|  36 | Gender Contradictions         | Examples such as `Marwan Naguib → Female` and `Nada Abdelaziz → Male`                                     | Investigated and corrected only when supported by additional evidence                |
|   37 | Malformed Institutional Email | Example: `@kayfa.io` for Tarek Gamal                                                                      | Reconstructed using institutional naming convention (`tarek.gamal@kayfa-student.io`) |


In [ ]:
students.head()

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
0,S0001,Hana Gamal,27.0,Male,Mansoura,hana.gamal@kayfa-student.io,G03,2025-12-14
1,S0002,Mona Abdelaziz,25.0,Female,Zagazig,mona.abdelaziz@kayfa-student.io,G06,2025-12-03
2,S0003,Menna Naguib,20.0,Male,Ismailia,menna.naguib@kayfa-student.io,G05,2025-12-17
3,S0004,Aya ElShafei,21.0,Male,Giza,aya.elshafei@kayfa-student.io,G02,2025-12-12
4,S0005,Habiba Mahmoud,24.0,Male,Giza,habiba.mahmoud@kayfa-student.io,G01,2025-12-14


In [ ]:
students.to_csv(
    "cleaned_data/students_clean.csv",
    index=False
)

## 3D · grades (flattened from JSON)

In [ ]:
print("=== GRADES ===")
print(f"Shape: {grades.shape}")
print(f"\nDuplicates: {grades.duplicated().sum()}")
print(f"\nNulls:\n{grades.isnull().sum()}")
print(f"\ntype values: {grades['type'].unique()}")
print(f"score range: {grades['score'].min()} - {grades['score'].max()}")
print(f"max_score values: {grades['max_score'].unique()}")

=== GRADES ===
Shape: (5502, 9)

Duplicates: 0

Nulls:
student_id          0
course_id           0
group_id            0
assessment_id       0
assessment_title    0
type                0
score               2
max_score           0
date                0
dtype: int64

type values: ['quiz' 'assignment' 'practical' 'exam']
score range: -10.0 - 187.0
max_score values: [100  10]


| Problem                     | Status         |
| --------------------------- | -------------- |
| 2 Missing Scores            | 🔍 Investigate |
| Negative Score (-10)        | 🔍 Investigate |
| Score > Max Score (187)     | 🔍 Investigate |
| max_score values = [100,10] | 🔍 Verify      |
| Duplicates                  | ✅              |


In [ ]:
grades[
    grades['student_id'] == 'S0001'
][[
    'assessment_id',
    'assessment_title',
    'type',
    'date'
]]

,assessment_id,assessment_title,type,date
0,C002-QZ,Quiz 1,quiz,2025-12-27
1,C002-QZ,Quiz 2,quiz,2026-01-10
2,C002-QZ,Quiz 3,quiz,2026-01-24
3,C002-QZ,Quiz 4,quiz,2026-02-07
4,C002-AS,Assignment 1,assignment,2026-02-21
5,C002-AS,Assignment 2,assignment,2026-03-07
6,C002-AS,Assignment 3,assignment,2026-03-21
7,C002-PR,Practical 1,practical,2026-04-04
8,C002-PR,Practical 2,practical,2026-04-18
9,C002-EX,Midterm Exam,exam,2026-05-02


In [ ]:


# SCORE > MAX_SCORE
impossible_scores = grades[
    grades['score'] > grades['max_score']
]

if len(impossible_scores) > 0:
    p = next_problem()
    log_problem(
        p,
        'grades.json',
        f"{len(impossible_scores)} records where score > max_score",
        "Investigate individually; possible data-entry errors"
    )

    print(
        impossible_scores[
            [
                'student_id',
                'assessment_id',
                'assessment_title',
                'score',
                'max_score'
            ]
        ]
    )

# NEGATIVE SCORES
negative_scores = grades[
    grades['score'] < 0
]

if len(negative_scores) > 0:
    p = next_problem()
    log_problem(
        p,
        'grades.json',
        f"{len(negative_scores)} negative scores",
        "Replace with NaN (true score unknown)"
    )

    print(
        negative_scores[
            [
                'student_id',
                'assessment_id',
                'assessment_title',
                'score'
            ]
        ]
    )



dup_assessments = grades.groupby(
    [
        'student_id',
        'assessment_id'
    ]
).size()

dups = dup_assessments[
    dup_assessments > 1
]

if len(dups) > 0:

    print(
        "\nINFO: Duplicate student-assessment_id combinations detected."
    )

    print(
        "Investigation showed assessment_id is reused "
        "for Quiz 1-4, Assignment 1-3, Practical 1-2."
    )

    print(
        "This is expected and NOT considered a data-quality issue."
    )

# TYPE consistency
print(
    f"\nAssessment type values: "
    f"{grades['type'].unique()}"
)

# DATE parsing
grades['date'] = pd.to_datetime(
    grades['date'],
    errors='coerce'
)

bad_grade_dates = grades[
    grades['date'].isna()
].shape[0]

if bad_grade_dates > 0:

    p = next_problem()

    log_problem(
        p,
        'grades.json',
        f"{bad_grade_dates} grade dates failed to parse",
        "Investigate original values"
    )

# FK — student exists
orphan_grades = grades[
    ~grades['student_id'].isin(
        students['student_id']
    )
]

if len(orphan_grades) > 0:

    p = next_problem()

    log_problem(
        p,
        'grades.json -> students.csv',
        f"{len(orphan_grades)} grade records for non-existent students",
        "Remove records after verification"
    )

# FK — course exists
orphan_courses = grades[
    ~grades['course_id'].isin(
        courses['course_id']
    )
]

if len(orphan_courses) > 0:

    p = next_problem()

    log_problem(
        p,
        'grades.json -> courses.csv',
        f"{len(orphan_courses)} records reference non-existent courses",
        "Investigate course mapping"
    )

# FK — group exists
orphan_groups = grades[
    ~grades['group_id'].isin(
        groups['group_id']
    )
]

if len(orphan_groups) > 0:

    p = next_problem()

    log_problem(
        p,
        'grades.json -> groups.csv',
        f"{len(orphan_groups)} records reference non-existent groups",
        "Investigate group mapping"
    )

🔴 Problem #17: 2 records where score > max_score
   student_id assessment_id assessment_title  score  max_score
12      S0002       C004-QZ           Quiz 2  187.0        100
44      S0005       C001-QZ           Quiz 1   78.2         10
🔴 Problem #18: 1 negative scores
  student_id assessment_id assessment_title  score
0      S0001       C002-QZ           Quiz 1  -10.0

INFO: Duplicate student-assessment_id combinations detected.
Investigation showed assessment_id is reused for Quiz 1-4, Assignment 1-3, Practical 1-2.
This is expected and NOT considered a data-quality issue.

Assessment type values: ['quiz' 'assignment' 'practical' 'exam']


In [ ]:
quiz = grades[
    (grades['assessment_id'] == 'C004-QZ') &
    (grades['assessment_title'] == 'Quiz 2')
]

quiz['score'].describe()

,score
count,56.000000
mean,76.444643
std,18.791729
min,49.400000
25%,66.575000
50%,74.650000
75%,84.025000
max,187.000000


In [ ]:
quiz.sort_values('score', ascending=False)[
    ['student_id','score']
].head(20)

,student_id,score
12,S0002,187.0
1917,S0175,95.3
1939,S0177,93.2
4711,S0429,92.0
4271,S0389,91.8
1763,S0161,90.8
267,S0025,90.6
3193,S0291,90.4
333,S0031,89.3
1213,S0111,89.0


In [ ]:
grades[
    (grades['assessment_id'] == 'C001-QZ') &
    (grades['assessment_title'] == 'Quiz 1')
]['max_score'].value_counts()

,count
max_score,
100,107
10,1


In [ ]:
grades[
    grades['score'] == 187
]

,student_id,course_id,group_id,assessment_id,assessment_title,type,score,max_score,date
12,S0002,C004,G06,C004-QZ,Quiz 2,quiz,187.0,100,2026-01-01


In [ ]:
grades[
    grades['student_id'] == 'S0002'
][[
    'assessment_title',
    'score',
    'max_score'
]]

,assessment_title,score,max_score
11,Quiz 1,97.1,100
12,Quiz 2,187.0,100
13,Quiz 3,93.8,100
14,Quiz 4,90.4,100
15,Assignment 1,90.6,100
16,Assignment 2,78.3,100
17,Assignment 3,72.9,100
18,Practical 1,82.1,100
19,Practical 2,72.0,100
20,Midterm Exam,83.7,100


In [ ]:
grades[
    grades['student_id'] == 'S0002'
]['score'].describe()

,score
count,11.000000
mean,94.145455
std,31.852829
min,72.000000
25%,80.200000
50%,87.700000
75%,92.200000
max,187.000000


In [ ]:
student_median = grades.loc[
    (grades['student_id'] == 'S0002') &
    (grades['score'] <= grades['max_score']),
    'score'
].median()

student_median

85.7

In [ ]:
grades.loc[12, 'score'] = student_median

Replaced impossible score (187) with the student's median performance score due to lack of reliable evidence for the original value.

In [ ]:
grades[
    grades['score'] == -10
]

,student_id,course_id,group_id,assessment_id,assessment_title,type,score,max_score,date
0,S0001,C002,G03,C002-QZ,Quiz 1,quiz,-10.0,100,2025-12-27


In [ ]:
grades[
    (grades['assessment_id'] == 'C002-QZ') &
    (grades['assessment_title'] == 'Quiz 1')
][['score','max_score']].describe()

,score,max_score
count,120.000000,120.0
mean,71.850000,100.0
std,12.781433,0.0
min,-10.000000,100.0
25%,65.475000,100.0
50%,71.650000,100.0
75%,79.700000,100.0
max,100.000000,100.0


In [ ]:
grades[
    grades['student_id'] == 'S0001'
][[
    'assessment_title',
    'score'
]]

,assessment_title,score
0,Quiz 1,-10.0
1,Quiz 2,69.1
2,Quiz 3,84.6
3,Quiz 4,88.2
4,Assignment 1,67.4
5,Assignment 2,61.8
6,Assignment 3,63.4
7,Practical 1,79.3
8,Practical 2,86.8
9,Midterm Exam,87.2


In [ ]:
student_scores = grades.loc[
    (grades['student_id'] == 'S0001') &
    (grades['score'] >= 0),
    'score'
]

grades.loc[
    (grades['student_id'] == 'S0001') &
    (grades['score'] < 0),
    'score'
] = student_scores.median()

In [ ]:
grades[
    grades['max_score'] == 10
]

,student_id,course_id,group_id,assessment_id,assessment_title,type,score,max_score,date
44,S0005,C001,G01,C001-QZ,Quiz 1,quiz,78.2,10,2025-12-27


In [ ]:
grades.loc[
    (grades['assessment_id'] == 'C001-QZ') &
    (grades['assessment_title'] == 'Quiz 1') &
    (grades['max_score'] == 10),
    'max_score'
] = 100

| Problem # | Issue                                                               | Fix                                                                                                |
| --------- | ------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------- |
|   38   | Incorrect max_score value (10 instead of 100) for one Quiz 1 record | Updated max_score from 10 to 100 based on consistency analysis of 107 matching assessment records. |


In [ ]:
grades['score'].isna().sum()

np.int64(2)

In [ ]:
grades[grades['score'].isna()]

,student_id,course_id,group_id,assessment_id,assessment_title,type,score,max_score,date
25,S0003,C003,G05,C003-QZ,Quiz 4,quiz,NaN,100,2026-02-09
37,S0004,C001,G02,C001-AS,Assignment 1,assignment,NaN,100,2026-02-20


In [ ]:
grades[
    grades['student_id'] == 'S0003'
][['assessment_title','score']]

,assessment_title,score
22,Quiz 1,62.1
23,Quiz 2,75.9
24,Quiz 3,73.8
25,Quiz 4,NaN
26,Assignment 1,81.3
27,Assignment 2,66.0
28,Assignment 3,63.5
29,Practical 1,66.4
30,Practical 2,62.2
31,Midterm Exam,78.3


In [ ]:
grades[
    grades['student_id'] == 'S0004'
][['assessment_title','score']]

,assessment_title,score
33,Quiz 1,86.7
34,Quiz 2,77.6
35,Quiz 3,77.6
36,Quiz 4,89.9
37,Assignment 1,NaN
38,Assignment 2,75.7
39,Assignment 3,79.9
40,Practical 1,80.3
41,Practical 2,89.5
42,Midterm Exam,80.0


In [ ]:
for sid in ['S0003', 'S0004']:

    median_score = grades.loc[
        (grades['student_id'] == sid) &
        (grades['score'].notna()),
        'score'
    ].median()

    grades.loc[
        (grades['student_id'] == sid) &
        (grades['score'].isna()),
        'score'
    ] = median_score

| Problem # | Issue                                  | Fix                                                                      |
| --------- | -------------------------------------- | ------------------------------------------------------------------------ |
|  39   | Missing score for Quiz 4 (S0003)       | Retained as missing due to insufficient evidence to infer original value |
|   40  | Missing score for Assignment 1 (S0004) | Retained as missing due to insufficient evidence to infer original value |


In [ ]:
import re

print("=" * 60)
print("FINAL GRADES VALIDATION")
print("=" * 60)

print(f"\nRows: {len(grades)}")

# 1. Nulls
print("\n1. NULL VALUES")
print(grades.isnull().sum())

# 2. Scores
print("\n2. SCORES")

invalid_negative = grades[grades['score'] < 0]

invalid_over_max = grades[
    grades['score'] > grades['max_score']
]

print(f"Negative scores: {len(invalid_negative)}")
print(f"Scores > max_score: {len(invalid_over_max)}")

# 3. Max Score
print("\n3. MAX SCORE")

print("Unique max_score values:")
print(sorted(grades['max_score'].unique()))

# 4. Assessment Types
print("\n4. ASSESSMENT TYPES")

print("Unique types:")
print(grades['type'].unique())

# 5. Student FK
print("\n5. STUDENT REFERENCES")

orphan_students = grades[
    ~grades['student_id'].isin(students['student_id'])
]

print(f"Invalid student_ids: {len(orphan_students)}")

# 6. Group FK
print("\n6. GROUP REFERENCES")

orphan_groups = grades[
    ~grades['group_id'].isin(groups['group_id'])
]

print(f"Invalid group_ids: {len(orphan_groups)}")

# 7. Course FK
print("\n7. COURSE REFERENCES")

orphan_courses = grades[
    ~grades['course_id'].isin(courses['course_id'])
]

print(f"Invalid course_ids: {len(orphan_courses)}")

# 8. Dates
print("\n8. DATES")

bad_dates = grades[
    grades['date'].isna()
]

print(f"Invalid dates: {len(bad_dates)}")

# 9. Duplicate Full Rows
print("\n9. DUPLICATE ROWS")

print(
    f"Duplicate rows: {grades.duplicated().sum()}"
)

# 10. Assessment Consistency
print("\n10. ASSESSMENT CONSISTENCY")

assessment_check = (
    grades.groupby(
        ['assessment_id', 'assessment_title']
    )
    .size()
)

print(
    f"Unique assessment combinations: {len(assessment_check)}"
)

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

FINAL GRADES VALIDATION

Rows: 5502

1. NULL VALUES
student_id          0
course_id           0
group_id            0
assessment_id       0
assessment_title    0
type                0
score               0
max_score           0
date                0
dtype: int64

2. SCORES
Negative scores: 0
Scores > max_score: 0

3. MAX SCORE
Unique max_score values:
[np.int64(100)]

4. ASSESSMENT TYPES
Unique types:
['quiz' 'assignment' 'practical' 'exam']

5. STUDENT REFERENCES
Invalid student_ids: 0

6. GROUP REFERENCES
Invalid group_ids: 0

7. COURSE REFERENCES
Invalid course_ids: 0

8. DATES
Invalid dates: 0

9. DUPLICATE ROWS
Duplicate rows: 0

10. ASSESSMENT CONSISTENCY
Unique assessment combinations: 78

VALIDATION COMPLETE


In [ ]:
# Save cleaned grades dataset

grades.to_csv(
    "cleaned_data/grades_clean.csv",
    index=False
)

print("✓ grades_clean.csv saved successfully")

✓ grades_clean.csv saved successfully


## 3E · engagement_events.csv

In [ ]:
print("=== ENGAGEMENT ===")
print(f"Shape: {engagement.shape}")
print(f"\nDuplicates: {engagement.duplicated().sum()}")
print(f"\nNulls:\n{engagement.isnull().sum()}")
print(f"\nevent_type values: {engagement['event_type'].unique()}")
print(f"device values: {engagement['device'].unique()}")
print(f"duration_seconds range: {engagement['duration_seconds'].min()} - {engagement['duration_seconds'].max()}")

=== ENGAGEMENT ===
Shape: (30866, 6)

Duplicates: 8

Nulls:
event_id                0
student_id              0
event_type              0
event_datetime          0
duration_seconds    22049
device                  0
dtype: int64

event_type values: ['quiz_attempt' 'video_watch' 'forum_post' 'login' 'resource_download']
device values: ['mobile' 'web']
duration_seconds range: -120.0 - 99999.0


In [ ]:
# Engagement — deep checks

# DURATION only for video_watch
non_video_with_dur = engagement[
    (engagement['event_type'] != 'video_watch') &
    (engagement['duration_seconds'].notna()) &
    (engagement['duration_seconds'] > 0)
]
if len(non_video_with_dur) > 0:
    p = next_problem()
    log_problem(p, 'engagement_events.csv',
               f"{len(non_video_with_dur)} non-video events have duration_seconds",
               "Set duration_seconds to NaN for non-video events")

# NEGATIVE DURATION
negative_dur = engagement[engagement['duration_seconds'] < 0]
if len(negative_dur) > 0:
    p = next_problem()
    log_problem(p, 'engagement_events.csv',
               f"{len(negative_dur)} negative duration_seconds",
               "Take absolute value or NaN")

# EXTREME DURATION (> 8 hours)
extreme_dur = engagement[engagement['duration_seconds'] > 28800]
if len(extreme_dur) > 0:
    p = next_problem()
    log_problem(p, 'engagement_events.csv',
               f"{len(extreme_dur)} duration > 8 hours — likely errors",
               "Cap at reasonable max or NaN")

# DATETIME
engagement['event_datetime'] = pd.to_datetime(engagement['event_datetime'], errors='coerce')

# FK
orphan_eng = engagement[~engagement['student_id'].isin(students['student_id'])]
if len(orphan_eng) > 0:
    p = next_problem()
    log_problem(p, 'engagement_events.csv -> students.csv',
               f"{len(orphan_eng)} events for non-existent students",
               "Drop")

# DEVICE inconsistency
print(f"\nDevice unique: {engagement['device'].unique()}")

# DUPLICATE event_ids
dup_events = engagement[engagement['event_id'].duplicated(keep=False)]
if len(dup_events) > 0:
    p = next_problem()
    log_problem(p, 'engagement_events.csv',
               f"{len(dup_events)} duplicate event_ids",
               "Keep first occurrence")

🔴 Problem #19: 2 negative duration_seconds
🔴 Problem #20: 1 duration > 8 hours — likely errors
🔴 Problem #21: 1 events for non-existent students

Device unique: ['mobile' 'web']
🔴 Problem #22: 16 duplicate event_ids


In [ ]:
engagement[
    engagement['duration_seconds'] > 28800
]

,event_id,student_id,event_type,event_datetime,duration_seconds,device
3,EV000004,S0001,video_watch,2026-04-08 17:16:22,99999.0,web


In [ ]:
engagement[
    engagement['duration_seconds'] < 0
]

,event_id,student_id,event_type,event_datetime,duration_seconds,device
1,EV000002,S0001,video_watch,2026-04-15 08:24:37,-120.0,web
30857,EVBAD02,S0001,video_watch,2027-06-01 10:00:00,-120.0,web


In [ ]:
engagement[
    ~engagement['student_id'].isin(
        students['student_id']
    )
]

,event_id,student_id,event_type,event_datetime,duration_seconds,device
30856,EVBAD01,S8888,quiz_attempt,2026-05-13 11:45:14,NaN,mobile


In [ ]:
submissions[
    submissions['submitted_at'].isna()
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
9,SUB00010,S0004,C001,C001-AS,2026-02-20 23:59:00,NaT,False,105,1


In [ ]:
submissions[
    submissions['time_spent_minutes'] < 0
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
5,SUB00006,S0002,C004,C004-AS,2026-03-16 23:59:00,2026-03-16 00:03:00,False,-40,3


In [ ]:
submissions.sort_values(
    'time_spent_minutes',
    ascending=False
).head(20)

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
1020,SUB01021,S0341,C002,C002-AS,2026-02-16 23:59:00,2026-02-17 10:38:00,True,255,2
37,SUB00038,S0013,C005,C005-AS,2026-03-04 23:59:00,2026-03-05 08:10:00,True,230,1
679,SUB00680,S0227,C001,C001-AS,2026-03-09 23:59:00,2026-03-09 10:59:00,False,227,1
758,SUB00759,S0253,C002,C002-AS,2026-03-20 23:59:00,2026-03-20 22:26:00,False,227,1
708,SUB00709,S0237,C006,C006-AS,2026-02-18 23:59:00,2026-02-17 23:21:00,False,226,2
245,SUB00246,S0082,C006,C006-AS,2026-03-23 23:59:00,2026-03-23 02:53:00,False,224,2
1478,SUB01479,S0493,C001,C001-AS,2026-03-16 23:59:00,2026-03-16 08:15:00,False,222,1
78,SUB00079,S0027,C002,C002-AS,2026-02-14 23:59:00,2026-02-15 03:46:00,True,222,1
545,SUB00546,S0182,C001,C001-AS,2026-03-19 23:59:00,2026-03-19 10:15:00,False,217,1
458,SUB00459,S0153,C005,C005-AS,2026-03-18 23:59:00,2026-03-18 08:49:00,False,217,1


In [ ]:
submissions[
    submissions.duplicated(
        keep=False
    )
].sort_values(
    'submission_id'
)

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
88,SUB00089,S0030,C006,C006-AS,2026-03-09 23:59:00,2026-03-09 23:27:00,False,170,1
1503,SUB00089,S0030,C006,C006-AS,2026-03-09 23:59:00,2026-03-09 23:27:00,False,170,1
488,SUB00489,S0163,C001,C001-AS,2026-03-19 23:59:00,2026-03-19 07:11:00,False,130,1
1502,SUB00489,S0163,C001,C001-AS,2026-03-19 23:59:00,2026-03-19 07:11:00,False,130,1
1240,SUB01241,S0414,C005,C005-AS,2026-03-01 23:59:00,2026-03-01 21:04:00,False,117,1
1501,SUB01241,S0414,C005,C005-AS,2026-03-01 23:59:00,2026-03-01 21:04:00,False,117,1


In [ ]:
submissions[
    submissions.duplicated(
        keep=False
    )
].sort_values(
    'submission_id'
)

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
88,SUB00089,S0030,C006,C006-AS,2026-03-09 23:59:00,2026-03-09 23:27:00,False,170,1
1503,SUB00089,S0030,C006,C006-AS,2026-03-09 23:59:00,2026-03-09 23:27:00,False,170,1
488,SUB00489,S0163,C001,C001-AS,2026-03-19 23:59:00,2026-03-19 07:11:00,False,130,1
1502,SUB00489,S0163,C001,C001-AS,2026-03-19 23:59:00,2026-03-19 07:11:00,False,130,1
1240,SUB01241,S0414,C005,C005-AS,2026-03-01 23:59:00,2026-03-01 21:04:00,False,117,1
1501,SUB01241,S0414,C005,C005-AS,2026-03-01 23:59:00,2026-03-01 21:04:00,False,117,1


In [ ]:
submissions[
    ~submissions['student_id'].isin(
        students['student_id']
    )
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts


In [ ]:
engagement.groupby('event_type')['duration_seconds'].apply(
    lambda x: x.isna().sum()
)

,duration_seconds
event_type,
forum_post,2240
login,11052
quiz_attempt,4318
resource_download,4439
video_watch,0


In [ ]:
engagement.groupby('event_type')['duration_seconds'].agg(
    ['count','size']
)

,count,size
event_type,,
forum_post,0,2240
login,0,11052
quiz_attempt,0,4318
resource_download,0,4439
video_watch,8817,8817


In [ ]:
# remove invalid student
engagement = engagement[
    engagement['student_id'].isin(students['student_id'])
]

# negative durations
engagement.loc[
    engagement['duration_seconds'] < 0,
    'duration_seconds'
] = np.nan

# unrealistic duration
engagement.loc[
    engagement['duration_seconds'] > 28800,
    'duration_seconds'
] = np.nan

# replace anomalies using video median
video_median = engagement.loc[
    (engagement['event_type'] == 'video_watch') &
    (engagement['duration_seconds'].notna()),
    'duration_seconds'
].median()

engagement.loc[
    (engagement['event_type'] == 'video_watch') &
    (engagement['duration_seconds'].isna()),
    'duration_seconds'
] = video_median

# remove duplicate ids
engagement = engagement.drop_duplicates(
    subset=['event_id'],
    keep='first'
)

In [ ]:
engagement['duration_seconds'] = (
    engagement['duration_seconds']
    .fillna(0)
)

In [ ]:
print("="*60)
print("FINAL ENGAGEMENT VALIDATION")
print("="*60)

print("\nRows:", len(engagement))

print("\n1. EVENT IDS")
print("Duplicate event_ids:",
      engagement['event_id'].duplicated().sum())

print("\n2. STUDENT REFERENCES")
print("Invalid students:",
      (~engagement['student_id'].isin(
          students['student_id']
      )).sum())

print("\n3. DURATION")
print("Negative durations:",
      (engagement['duration_seconds'] < 0).sum())

print("Durations > 8h:",
      (engagement['duration_seconds'] > 28800).sum())

print("\n4. EVENT TYPES")
print(engagement['event_type'].unique())

print("\n5. DEVICES")
print(engagement['device'].unique())

print("\n6. DUPLICATE ROWS")
print("Duplicate rows:",
      engagement.duplicated().sum())

print("="*60)

FINAL ENGAGEMENT VALIDATION

Rows: 30857

1. EVENT IDS
Duplicate event_ids: 0

2. STUDENT REFERENCES
Invalid students: 0

3. DURATION
Negative durations: 0
Durations > 8h: 0

4. EVENT TYPES
['quiz_attempt' 'video_watch' 'forum_post' 'login' 'resource_download']

5. DEVICES
['mobile' 'web']

6. DUPLICATE ROWS
Duplicate rows: 0


In [ ]:
engagement.to_csv(
    "cleaned_data/engagement_clean.csv",
    index=False
)

login              → duration never recorded
forum_post         → duration never recorded
resource_download  → duration never recorded
quiz_attempt       → duration never recorded
video_watch        → duration always recorded

| Problem # | Issue                          | Resolution                                                                                                                                  |
| --------- | ------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------- |
| 41     | 22,049 missing duration values | Investigation showed duration is not applicable for login, forum_post, resource_download, and quiz_attempt events. Values retained as null. |


| Problem # | Issue                                                     | Records Affected | Investigation                                                                                                                                                                                                                                                                                          | Resolution                                                                                                                                                |
| --------- | --------------------------------------------------------- | ---------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------- |
|      42 | Negative `duration_seconds` (-120)                        | 2                | Both records were `video_watch` events. Negative watch time is impossible and indicates a data-entry/system error.                                                                                                                                                                                     | Converted to `NaN` and imputed using the median duration of valid `video_watch` events.                                                                   |
|   43    | Unrealistic `duration_seconds` (99,999 sec)               | 1                | Equivalent to approximately 27.8 hours of continuous video watching, which is not realistic for a single engagement event.                                                                                                                                                                             | Converted to `NaN` and imputed using the median duration of valid `video_watch` events.                                                                   |
|    44  | Engagement event linked to non-existent student (`S8888`) | 1                | Student ID did not exist in the cleaned Students dataset, violating referential integrity.                                                                                                                                                                                                             | Removed the orphan record.                                                                                                                                |
|   45  | Duplicate `event_id` values                               | 16 rows          | Event IDs are intended to be unique identifiers. Investigation confirmed duplicated event records.                                                                                                                                                                                                     | Retained the first occurrence and removed duplicate records.                                                                                              |
|    46 | Missing `duration_seconds` values                         | 22,047          | Analysis by event type showed that all missing durations belonged to `login`, `forum_post`, `resource_download`, and `quiz_attempt` events. `video_watch` events had complete duration data (8,817/8,817 populated). This indicates duration tracking is only applicable to video-watching activities. | No correction required. Values were retained as NULL because duration is structurally not applicable to these event types rather than being missing data. |


## 3F · assignment_submissions.csv

In [ ]:
print("=== SUBMISSIONS ===")
print(f"Shape: {submissions.shape}")
print(f"\nDuplicates: {submissions.duplicated().sum()}")
print(f"\nNulls:\n{submissions.isnull().sum()}")
print(f"\nis_late values: {submissions['is_late'].unique()}")
print(f"time_spent_minutes range: {submissions['time_spent_minutes'].min()} - {submissions['time_spent_minutes'].max()}")
print(f"attempts range: {submissions['attempts'].min()} - {submissions['attempts'].max()}")

=== SUBMISSIONS ===
Shape: (1504, 9)

Duplicates: 3

Nulls:
submission_id         0
student_id            0
course_id             0
assessment_id         0
deadline              0
submitted_at          1
is_late               0
time_spent_minutes    0
attempts              0
dtype: int64

is_late values: [False  True]
time_spent_minutes range: 10.0 - 255.0
attempts range: 1 - 3


In [ ]:
# Submissions — deep checks

# PARSE DATETIMES
submissions['deadline'] = pd.to_datetime(submissions['deadline'], errors='coerce')
submissions['submitted_at'] = pd.to_datetime(submissions['submitted_at'], errors='coerce')

# THE KEY CHECK: verify is_late against actual timestamps
submissions['computed_is_late'] = submissions['submitted_at'] > submissions['deadline']
mismatch = submissions[submissions['is_late'] != submissions['computed_is_late']]
if len(mismatch) > 0:
    p = next_problem()
    log_problem(p, 'assignment_submissions.csv',
               f"{len(mismatch)} rows where is_late flag doesn't match timestamps",
               "Override is_late with computed value")
    print(mismatch[['submission_id', 'deadline', 'submitted_at', 'is_late', 'computed_is_late']].head(10))
    submissions['is_late'] = submissions['computed_is_late']

# NEGATIVE time_spent
neg_time = submissions[submissions['time_spent_minutes'] < 0]
if len(neg_time) > 0:
    p = next_problem()
    log_problem(p, 'assignment_submissions.csv',
               f"{len(neg_time)} negative time_spent_minutes",
               "Take absolute value or NaN")

# ZERO ATTEMPTS
zero_att = submissions[submissions['attempts'] < 1]
if len(zero_att) > 0:
    p = next_problem()
    log_problem(p, 'assignment_submissions.csv',
               f"{len(zero_att)} submissions with attempts < 1",
               "Set to 1 (minimum possible)")

# FK
orphan_subs = submissions[~submissions['student_id'].isin(students['student_id'])]
if len(orphan_subs) > 0:
    p = next_problem()
    log_problem(p, 'assignment_submissions.csv -> students.csv',
               f"{len(orphan_subs)} submissions for non-existent students",
               "Drop")

# Clean up temp column
submissions.drop(columns=['computed_is_late'], inplace=True, errors='ignore')

In [ ]:
submissions['is_late'] = (
    submissions['submitted_at'] >
    submissions['deadline']
)

In [ ]:
mask = (
    (submissions['submitted_at'].notna()) &
    (
        submissions['is_late'] !=
        (submissions['submitted_at'] > submissions['deadline'])
    )
)

submissions.loc[mask, 'is_late'] = (
    submissions.loc[mask, 'submitted_at'] >
    submissions.loc[mask, 'deadline']
)

In [ ]:
submissions[
    submissions['time_spent_minutes'] < 0
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
5,SUB00006,S0002,C004,C004-AS,2026-03-16 23:59:00,2026-03-16 00:03:00,False,-40,3


In [ ]:
submissions[
    submissions['student_id'] == 'S0002'
].sort_values('submitted_at')

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
3,SUB00004,S0002,C004,C004-AS,2026-02-15 23:59:00,2026-02-15 16:43:00,False,73.0,2
4,SUB00005,S0002,C004,C004-AS,2026-03-02 23:59:00,2026-03-02 16:37:00,False,91.0,3
5,SUB00006,S0002,C004,C004-AS,2026-03-16 23:59:00,2026-03-16 00:03:00,False,NaN,3


In [ ]:
submissions.loc[
    submissions['time_spent_minutes'] < 0,
    'time_spent_minutes'
] = np.nan

In [ ]:
submissions['is_late'] = (
    submissions['submitted_at'] > submissions['deadline']
)

In [ ]:
print("Duplicates:", submissions.duplicated().sum())

print("\nNulls:")
print(submissions.isna().sum())

print("\nLate flag check:")
print(
    (
        submissions['is_late']
        !=
        (submissions['submitted_at'] > submissions['deadline'])
    ).sum()
)

Duplicates: 3

Nulls:
submission_id         0
student_id            0
course_id             0
assessment_id         0
deadline              0
submitted_at          1
is_late               0
time_spent_minutes    1
attempts              0
dtype: int64

Late flag check:
0


In [ ]:
submissions[
    submissions['submitted_at'].isna()
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
9,SUB00010,S0004,C001,C001-AS,2026-02-20 23:59:00,NaT,False,105.0,1


In [ ]:
submissions[
    submissions['time_spent_minutes'].isna()
]

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts
5,SUB00006,S0002,C004,C004-AS,2026-03-16 23:59:00,2026-03-16 00:03:00,False,NaN,3


In [ ]:
course_median = submissions.loc[
    (submissions['course_id'] == 'C004') &
    (submissions['time_spent_minutes'].notna()),
    'time_spent_minutes'
].median()

course_median

117.0

In [ ]:
submissions.loc[
    submissions['submission_id'] == 'SUB00006',
    'time_spent_minutes'
] = course_median

In [ ]:
submissions.to_csv(
    "cleaned_data/submissions_clean.csv",
    index=False
)

print("submissions_clean.csv saved successfully")

submissions_clean.csv saved successfully


| Problem # | Issue                               | Records Affected | Investigation                                                                                                                                                                                             | Resolution                                                                                            |
| --------- | ----------------------------------- | ---------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- |
|    47 | Incorrect `is_late` flag            | 2                | `submitted_at` occurred before or exactly at the deadline, but `is_late=True`. Comparison with `deadline` proved the flag was wrong.                                                                      | Recomputed `is_late` using `submitted_at > deadline`.                                                 |
|    48 | Negative `time_spent_minutes` (-40) | 1                | Negative completion time is impossible. Review of the student's other submissions (73 and 91 minutes) confirmed the value was invalid, but the true value could not be determined.                        | Converted to `NaN` and imputed using the median valid completion time for the same course/assessment. |
|49    | Missing `submitted_at` timestamp    | 1                | Submission record contained a valid student, assessment, attempts count, and time spent, indicating the submission occurred. However, the exact submission timestamp could not be reconstructed reliably. | Retained as `NULL` and documented as unrecoverable missing information.                               |
|  50    | Duplicate submission records        | 3 duplicate rows | Investigation showed the rows were exact duplicates (`SUB00089`, `SUB00489`, `SUB01241`).                                                                                                                 | Removed duplicate rows using `drop_duplicates()`.                                                     |


## 3G · concepts_performance.csv

In [ ]:
concepts.head()

,record_id,student_id,course_id,assessment_id,question_no,concept_id,concept_name,score_pct,mastery_status,timestamp
0,CP000001,S0001,C002,C002-QZ,1,C002-K01,Variables & Types,80.0,passed,2025-12-27 23:06:24
1,CP000002,S0001,C002,C002-QZ,2,C002-K03,Functions,82.0,passed,2025-12-27 23:06:24
2,CP000003,S0001,C002,C002-QZ,3,C002-K06,File I/O,83.2,passed,2025-12-27 23:06:24
3,CP000004,S0001,C002,C002-QZ,4,C002-K05,Recursion,49.3,failed,2025-12-27 23:06:24
4,CP000005,S0001,C002,C002-QZ,1,C002-K02,Control Flow,59.4,failed,2026-01-10 23:06:24


In [ ]:
print("=== CONCEPTS ===")
print(f"Shape: {concepts.shape}")
print(f"\nDuplicates: {concepts.duplicated().sum()}")
print(f"\nNulls:\n{concepts.isnull().sum()}")
print(f"\nmastery_status values: {concepts['mastery_status'].unique()}")
print(f"score_pct range: {concepts['score_pct'].min()} - {concepts['score_pct'].max()}")

=== CONCEPTS ===
Shape: (12008, 10)

Duplicates: 5

Nulls:
record_id         0
student_id        0
course_id         0
assessment_id     0
question_no       0
concept_id        0
concept_name      0
score_pct         0
mastery_status    0
timestamp         0
dtype: int64

mastery_status values: ['passed' 'failed']
score_pct range: -33.0 - 142.0


In [ ]:
# Concepts — deep checks

# SCORE_PCT outside 0-100
bad_pct = concepts[(concepts['score_pct'] < 0) | (concepts['score_pct'] > 100)]
if len(bad_pct) > 0:
    p = next_problem()
    log_problem(p, 'concepts_performance.csv',
               f"{len(bad_pct)} score_pct values outside 0-100",
               "Clip to 0-100")

# MASTERY_STATUS vs SCORE_PCT consistency
passed_low = concepts[(concepts['mastery_status'] == 'passed') & (concepts['score_pct'] < 30)]
failed_high = concepts[(concepts['mastery_status'] == 'failed') & (concepts['score_pct'] > 80)]
if len(passed_low) > 0 or len(failed_high) > 0:
    p = next_problem()
    log_problem(p, 'concepts_performance.csv',
               f"{len(passed_low)} 'passed' with score<30, {len(failed_high)} 'failed' with score>80",
               "Flag — status may be flipped")

# DUPLICATE record_ids
dup_rec = concepts[concepts['record_id'].duplicated(keep=False)]
if len(dup_rec) > 0:
    p = next_problem()
    log_problem(p, 'concepts_performance.csv',
               f"{len(dup_rec)} duplicate record_ids",
               "Keep first occurrence")

# FK
orphan_con = concepts[~concepts['student_id'].isin(students['student_id'])]
if len(orphan_con) > 0:
    p = next_problem()
    log_problem(p, 'concepts_performance.csv -> students.csv',
               f"{len(orphan_con)} concept records for non-existent students",
               "Drop")

# TIMESTAMP parsing
concepts['timestamp'] = pd.to_datetime(concepts['timestamp'], errors='coerce')
bad_ts = concepts[concepts['timestamp'].isna()].shape[0]
if bad_ts > 0:
    p = next_problem()
    log_problem(p, 'concepts_performance.csv',
               f"{bad_ts} timestamps failed to parse",
               "Investigate")

/tmp/ipykernel_14127/4136461975.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  concepts['timestamp'] = pd.to_datetime(concepts['timestamp'], errors='coerce')


In [ ]:
concepts[
    (concepts['score_pct'] < 0) |
    (concepts['score_pct'] > 100)
]

,record_id,student_id,course_id,assessment_id,question_no,concept_id,concept_name,score_pct,mastery_status,timestamp
12006,CPBAD02,S0009,C001,C001-QZ,1,C001-K05,Joins & Merges,-33.0,failed,2026-01-18 13:56:35
12007,CPBAD03,S0013,C005,C005-QZ,1,C005-K04,Funnel Analytics,142.0,failed,2026-02-03 05:45:29


In [ ]:
concepts[
    (
        (concepts['mastery_status'] == 'failed') &
        (concepts['score_pct'] > 80)
    )
]

,record_id,student_id,course_id,assessment_id,question_no,concept_id,concept_name,score_pct,mastery_status,timestamp
12007,CPBAD03,S0013,C005,C005-QZ,1,C005-K04,Funnel Analytics,142.0,failed,2026-02-03 05:45:29


In [ ]:
# Problem 25
concepts[
    (concepts['score_pct'] < 0) |
    (concepts['score_pct'] > 100)
]

# Problem 26
concepts[
    ((concepts['mastery_status']=='failed') & (concepts['score_pct']>80))
]

# Problem 27
concepts[
    concepts['record_id'].duplicated(keep=False)
].sort_values('record_id')

,record_id,student_id,course_id,assessment_id,question_no,concept_id,concept_name,score_pct,mastery_status,timestamp
247,CP000248,S0011,C006,C006-QZ,4,C006-K03,Overfitting & Regularization,56.5,failed,2026-01-05 21:54:11
12004,CP000248,S0011,C006,C006-QZ,4,C006-K03,Overfitting & Regularization,56.5,failed,2026-01-05 21:54:11
1857,CP001858,S0078,C002,C002-QZ,2,C002-K01,Variables & Types,54.3,failed,2026-01-25 08:48:21
12003,CP001858,S0078,C002,C002-QZ,2,C002-K01,Variables & Types,54.3,failed,2026-01-25 08:48:21
7451,CP007452,S0311,C005,C005-QZ,4,C005-K03,Paid Ads,68.5,passed,2026-01-24 08:41:31
12001,CP007452,S0311,C005,C005-QZ,4,C005-K03,Paid Ads,68.5,passed,2026-01-24 08:41:31
8370,CP008371,S0349,C004,C004-EX,3,C004-K01,Design Principles,66.4,passed,2026-05-02 08:19:54
12002,CP008371,S0349,C004,C004-EX,3,C004-K01,Design Principles,66.4,passed,2026-05-02 08:19:54
9063,CP009064,S0378,C001,C001-QZ,4,C001-K04,GroupBy & Aggregation,61.5,passed,2026-02-07 15:42:38
12000,CP009064,S0378,C001,C001-QZ,4,C001-K04,GroupBy & Aggregation,61.5,passed,2026-02-07 15:42:38


In [ ]:
concepts = concepts.copy()

concepts['timestamp'] = pd.to_datetime(
    concepts['timestamp'],
    errors='coerce'
)

In [ ]:
concepts.loc[
    (concepts['score_pct'] < 0) |
    (concepts['score_pct'] > 100),
    'score_pct'
] = np.nan

In [ ]:
concepts['score_pct'] = concepts.groupby(
    'concept_id'
)['score_pct'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
concepts.loc[
    concepts['score_pct'] >= 80,
    'mastery_status'
] = 'passed'

In [ ]:
concepts = concepts.drop_duplicates()

In [ ]:
print("Duplicates:", concepts.duplicated().sum())

print("\nNulls:")
print(concepts.isnull().sum())

print("\nInvalid scores:")
print(
    ((concepts['score_pct'] < 0) |
     (concepts['score_pct'] > 100)).sum()
)

print("\nMastery mismatches:")
print(
    ((concepts['mastery_status'] == 'failed') &
     (concepts['score_pct'] > 80)).sum()
)

Duplicates: 0

Nulls:
record_id         0
student_id        0
course_id         0
assessment_id     0
question_no       0
concept_id        0
concept_name      0
score_pct         0
mastery_status    0
timestamp         0
dtype: int64

Invalid scores:
0

Mastery mismatches:
0


In [ ]:
concepts.to_csv(
    "cleaned_data/concepts_clean.csv",
    index=False
)

print("concepts_clean.csv saved successfully")

concepts_clean.csv saved successfully


| Problem # | Issue                                   | Records Affected               | Investigation                                                                                                               | Resolution                                                                   |
| --------- | --------------------------------------- | ------------------------------ | --------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------- |
|  51  | `score_pct` outside valid range (0–100) | 2                              | Values -33 and 142 are impossible percentages and indicate data-entry errors. True values could not be determined reliably. | Replaced with `NaN` and imputed using the median score for the same concept. |
| 52| Inconsistent `mastery_status`           | 1                              | Record marked as `failed` despite having a score above the mastery threshold.                                               | Corrected `mastery_status` to match the score.                               |
|  53 | Duplicate `record_id` values            | 10 rows (5 duplicated records) | Investigation showed duplicated records were exact copies across all fields.                                                | Removed duplicate rows and retained one copy of each record.                 |


### Data Validation across files

In [ ]:
print("="*60)
print("FINAL CONCEPTS VALIDATION")
print("="*60)

print("\nRows:", len(concepts))

print("\nDuplicate rows:", concepts.duplicated().sum())

print("\nNulls:")
print(concepts.isnull().sum())

print("\nInvalid score_pct:")
print(((concepts['score_pct'] < 0) | (concepts['score_pct'] > 100)).sum())

print("\nInvalid mastery status:")
print(
    concepts[
        ~concepts['mastery_status'].isin(['passed', 'failed'])
    ].shape[0]
)

print("\nDuplicate record_ids:")
print(concepts['record_id'].duplicated().sum())

print("\nInvalid student_ids:")
print(
    (~concepts['student_id'].isin(students['student_id'])).sum()
)

print("\nInvalid course_ids:")
print(
    (~concepts['course_id'].isin(courses['course_id'])).sum()
)

print("\nInvalid timestamps:")
print(concepts['timestamp'].isna().sum())

print("\nValidation Complete")

FINAL CONCEPTS VALIDATION

Rows: 12003

Duplicate rows: 0

Nulls:
record_id         0
student_id        0
course_id         0
assessment_id     0
question_no       0
concept_id        0
concept_name      0
score_pct         0
mastery_status    0
timestamp         0
dtype: int64

Invalid score_pct:
0

Invalid mastery status:
0

Duplicate record_ids:
0

Invalid student_ids:
0

Invalid course_ids:
0

Invalid timestamps:
0

Validation Complete


In [ ]:
import shutil

# Create ZIP file
shutil.make_archive(
    "cleaned_data",
    "zip",
    "cleaned_data"
)

print("cleaned_data.zip created successfully")

cleaned_data.zip created successfully


In [ ]:
from google.colab import files

files.download("cleaned_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
# ═══════════════════════════════════════════
# SECTION 4 · CROSS-FILE LOGIC CHECKS
# ═══════════════════════════════════════════



In [ ]:
import zipfile

zip_path = "cleaned_data.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("clean_data")

print("Extracted successfully")

Extracted successfully


In [ ]:
import pandas as pd
import os

path = "clean_data"

students = pd.read_csv(f"{path}/students_clean.csv")
courses = pd.read_csv(f"{path}/courses.csv")
groups = pd.read_csv(f"{path}/groups_clean.csv")
grades = pd.read_csv(f"{path}/grades_clean.csv")
attendance = pd.read_csv(f"{path}/attendance_clean.csv")
engagement = pd.read_csv(f"{path}/engagement_clean.csv")
submissions = pd.read_csv(f"{path}/submissions_clean.csv")
concepts = pd.read_csv(f"{path}/concepts_clean.csv")

In [ ]:
datasets = {
    "students": students,
    "courses": courses,
    "groups": groups,
    "grades": grades,
    "attendance": attendance,
    "engagement": engagement,
    "submissions": submissions,
    "concepts": concepts
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

students: (500, 8)
courses: (7, 8)
groups: (10, 7)
grades: (5502, 9)
attendance: (13010, 7)
engagement: (30857, 6)
submissions: (1504, 9)
concepts: (12003, 10)


In [ ]:

# Attendance rate
att_total = (
    attendance.groupby('student_id')
    .size()
    .reset_index(name='att_total')
)

att_present = (
    attendance[attendance['status'] == 'attended']
    .groupby('student_id')
    .size()
    .reset_index(name='att_present')
)

att_rate = att_total.merge(att_present, on='student_id', how='left')
att_rate['att_present'] = att_rate['att_present'].fillna(0)

att_rate['attendance_rate'] = (
    att_rate['att_present'] / att_rate['att_total']
)

att_rate = att_rate[['student_id', 'attendance_rate']]


# Average grade
avg_grade = (
    grades.groupby('student_id')['score']
    .mean()
    .reset_index(name='avg_grade')
)


# Engagement metrics
eng_counts = (
    engagement.groupby('student_id')
    .agg(
        total_events=('event_id', 'count'),
        login_count=('event_type',
                     lambda x: (x == 'login').sum())
    )
    .reset_index()
)


# Video watch time only
video_watch = (
    engagement[
        engagement['event_type'] == 'video_watch'
    ]
    .groupby('student_id')['duration_seconds']
    .sum()
    .reset_index(name='video_watch_time')
)


# Failed concepts count
failed_concepts = (
    concepts[
        concepts['mastery_status'] == 'failed'
    ]
    .groupby('student_id')
    .size()
    .reset_index(name='num_failed_concepts')
)


# Average concept score
concept_scores = (
    concepts.groupby('student_id')
    .agg(
        avg_concept_score=('score_pct', 'mean')
    )
    .reset_index()
)


# Submission metrics
submission_stats = (
    submissions.groupby('student_id')
    .agg(
        total_submissions=('submission_id', 'count'),
        avg_time_spent=('time_spent_minutes', 'mean'),
        avg_attempts=('attempts', 'mean')
    )
    .reset_index()
)

print("✅ All helper aggregates created")

✅ All helper aggregates created


In [ ]:
print("students:", students.shape)
print("courses:", courses.shape)
print("groups:", groups.shape)

students: (500, 8)
courses: (7, 8)
groups: (10, 7)


In [ ]:


master = students.copy()

# Group information
master = master.merge(
    groups[
        ['group_id',
         'group_name',
         'course_id',
         'instructor']
    ],
    on='group_id',
    how='left'
)

# Course information
master = master.merge(
    courses[
        ['course_id',
         'course_name',
         'category',
         'difficulty_level']
    ],
    on='course_id',
    how='left'
)

# Performance
master = master.merge(att_rate, on='student_id', how='left')
master = master.merge(avg_grade, on='student_id', how='left')

# Engagement
master = master.merge(eng_counts, on='student_id', how='left')
master = master.merge(video_watch, on='student_id', how='left')

# Concepts
master = master.merge(failed_concepts, on='student_id', how='left')
master = master.merge(concept_scores, on='student_id', how='left')

# Submissions
master = master.merge(submission_stats, on='student_id', how='left')




In [ ]:
master.isnull().sum()

,0
student_id,0
full_name,0
age,4
gender,0
city,0
email,0
group_id,0
enrollment_date,0
group_name,0
course_id,0


In [ ]:
failed_concepts = (
    concepts[concepts['mastery_status'] == 'failed']
    .groupby('student_id')
    .size()
    .reset_index(name='num_failed_concepts')
)

In [ ]:
master = master.merge(
    failed_concepts,
    on='student_id',
    how='left'
)

In [ ]:
students_with_nan = master.loc[
    master['num_failed_concepts'].isna(),
    'student_id'
]

concepts[
    concepts['student_id'].isin(students_with_nan)
].head()

,record_id,student_id,course_id,assessment_id,question_no,concept_id,concept_name,score_pct,mastery_status,timestamp
24,CP000025,S0002,C004,C004-QZ,1,C004-K01,Design Principles,97.5,passed,2025-12-17 01:03:39
25,CP000026,S0002,C004,C004-QZ,2,C004-K05,Usability Testing,79.7,passed,2025-12-17 01:03:39
26,CP000027,S0002,C004,C004-QZ,3,C004-K04,Prototyping,93.0,passed,2025-12-17 01:03:39
27,CP000028,S0002,C004,C004-QZ,4,C004-K03,Color Theory,100.0,passed,2025-12-17 01:03:39
28,CP000029,S0002,C004,C004-QZ,1,C004-K05,Usability Testing,94.8,passed,2026-01-01 01:03:39


In [ ]:
print("MASTER COLUMNS:")
print(master.columns.tolist())

print("\nFAILED_CONCEPTS:")
print(failed_concepts.head())

print("\nFAILED_CONCEPTS COLUMNS:")
print(failed_concepts.columns.tolist())

MASTER COLUMNS:
['student_id', 'full_name', 'age', 'gender', 'city', 'email', 'group_id', 'enrollment_date', 'group_name', 'course_id', 'instructor', 'course_name', 'category', 'difficulty_level', 'attendance_rate', 'avg_grade', 'total_events', 'login_count', 'video_watch_time', 'num_failed_concepts_x', 'avg_concept_score', 'total_submissions', 'avg_time_spent', 'avg_attempts', 'num_failed_concepts_y', 'num_failed_concepts']

FAILED_CONCEPTS:
  student_id  num_failed_concepts
0      S0001                    6
1      S0003                    2
2      S0005                    7
3      S0006                   10
4      S0007                    6

FAILED_CONCEPTS COLUMNS:
['student_id', 'num_failed_concepts']


In [ ]:
(master['num_failed_concepts_x'] == master['num_failed_concepts']).all()

np.False_

In [ ]:
(master['num_failed_concepts_y'] == master['num_failed_concepts']).all()

np.False_

In [ ]:
master[
    [
        'num_failed_concepts_x',
        'num_failed_concepts_y',
        'num_failed_concepts'
    ]
].isna().sum()

,0
num_failed_concepts_x,37
num_failed_concepts_y,37
num_failed_concepts,37


In [ ]:
master = master.drop(
    columns=[
        'num_failed_concepts_x',
        'num_failed_concepts_y'
    ]
)

In [ ]:
master['num_failed_concepts'].isna().sum()

np.int64(37)

In [ ]:
students_with_nan = master.loc[
    master['num_failed_concepts'].isna(),
    'student_id'
]

concepts[
    concepts['student_id'].isin(students_with_nan)
].shape

(888, 10)

In [ ]:
master['num_failed_concepts'] = (
    master['num_failed_concepts']
    .fillna(0)
)

| Problem # | Issue                                                                      | Records Affected | Investigation                                                                                                                                                                                                                                                                                                   | Resolution                                                                     |
| --------- | -------------------------------------------------------------------------- | ---------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------ |
| 54   | Missing `num_failed_concepts` after merging concept-performance aggregates | 37 students      | Investigation showed that all 37 students existed in the `concepts` dataset (888 concept records found). However, these students had no records with `mastery_status='failed'`, so they were absent from the failed-concepts aggregation table. The missing values were therefore structural, not missing data. | Replaced `NaN` with `0` to indicate that the student has zero failed concepts. |


In [ ]:
# Sanity check

print(f"Unique students: {master['student_id'].nunique()}")
print(f"Unique groups: {master['group_id'].nunique()}")
print(f"Unique courses: {master['course_id'].nunique()}")
print(f"\nNull check:\n{master.isnull().sum()}")
print(f"\nDescribe:\n{master.describe()}")

# Verify no duplicate students
assert master['student_id'].nunique() == master.shape[0], \
    "DUPLICATE STUDENTS IN MASTER — check join keys!"
print("\n\u2705 No duplicate students — master is clean")

Unique students: 500
Unique groups: 10
Unique courses: 7

Null check:
student_id             0
full_name              0
age                    4
gender                 0
city                   0
email                  0
group_id               0
enrollment_date        0
group_name             0
course_id              0
instructor             0
course_name            0
category               0
difficulty_level       0
attendance_rate        0
avg_grade              0
total_events           0
login_count            0
video_watch_time       0
avg_concept_score      0
total_submissions      0
avg_time_spent         0
avg_attempts           0
num_failed_concepts    0
dtype: int64

Describe:
              age  attendance_rate   avg_grade  total_events  login_count  \
count  496.000000       500.000000  500.000000    500.000000    500.00000   
mean    21.439516         0.767934   70.504198     61.714000     22.10200   
std      3.032546         0.113983    8.042422     13.850049      6.02696  

#### Feature Engineering

In [ ]:
master['engagement_score'] = (
    master['login_count'] * 0.2 +
    master['attendance_rate'] * 100 * 0.3 +
    master['avg_grade'] * 0.3 +
    (master['video_watch_time'] / 1000) * 0.2
)

In [ ]:
master['risk_flag'] = (
    (master['attendance_rate'] < 0.60) |
    (master['avg_grade'] < 60) |
    (master['num_failed_concepts'] > 10)
)

In [ ]:
master['performance_band'] = pd.cut(
    master['avg_grade'],
    bins=[0,60,70,80,100],
    labels=['Low','Average','Good','Excellent']
)

In [ ]:
master.head()

,student_id,full_name,age,gender,city,email,group_id,enrollment_date,group_name,course_id,...,login_count,video_watch_time,avg_concept_score,total_submissions,avg_time_spent,avg_attempts,num_failed_concepts,engagement_score,risk_flag,performance_band
0,S0001,Hana Gamal,27.0,Male,Mansoura,hana.gamal@kayfa-student.io,G03,2025-12-14,Group 03 — C002,C002,...,16,12159.0,73.995833,3,119.000000,1.333333,6.0,53.060751,False,Good
1,S0002,Mona Abdelaziz,25.0,Female,Zagazig,mona.abdelaziz@kayfa-student.io,G06,2025-12-03,Group 06 — C004,C004,...,32,18570.0,87.950000,3,93.666667,2.666667,0.0,63.287217,False,Excellent
2,S0003,Menna Naguib,20.0,Male,Ismailia,menna.naguib@kayfa-student.io,G05,2025-12-17,Group 05 — C003,C003,...,27,11317.0,77.120833,3,127.000000,1.666667,2.0,54.440743,False,Good
3,S0004,Aya ElShafei,21.0,Male,Giza,aya.elshafei@kayfa-student.io,G02,2025-12-12,Group 02 — C001,C001,...,27,17301.0,76.475000,3,121.666667,1.333333,0.0,58.704361,False,Excellent
4,S0005,Habiba Mahmoud,24.0,Male,Giza,habiba.mahmoud@kayfa-student.io,G01,2025-12-14,Group 01 — C001,C001,...,17,11072.0,71.336000,3,99.333333,2.000000,7.0,48.766568,False,Good


In [ ]:
master.columns

Index(['student_id', 'full_name', 'age', 'gender', 'city', 'email', 'group_id',
       'enrollment_date', 'group_name', 'course_id', 'instructor',
       'course_name', 'category', 'difficulty_level', 'attendance_rate',
       'avg_grade', 'total_events', 'login_count', 'video_watch_time',
       'avg_concept_score', 'total_submissions', 'avg_time_spent',
       'avg_attempts', 'num_failed_concepts', 'engagement_score', 'risk_flag',
       'performance_band'],
      dtype='object')

In [ ]:
master.to_csv(
    "clean_data/master_student_dataset.csv",
    index=False
)

print("✅ Master dataset saved")

✅ Master dataset saved


### Issued I found --not everything --

| #  | Dataset        | Issue                                                                                                         |
| -- | -------------- | ------------------------------------------------------------------------------------------------------------- |
| 1  | Student        | Duplicate student IDs                                                                                         |
| 2  | Student        | Missing full names                                                                                            |
| 3  | Student        | Invalid age values                                                                                            |
| 4  | Student        | Inconsistent gender encoding                                                                                  |
| 5  | Student        | Invalid email formats                                                                                         |
| 6  | Student        | Invalid group references                                                                                      |
| 7  | Student        | Duplicate email addresses                                                                                     |
| 8  | Student        | Identity conflicts across student records                                                                     |
| 9  | Student        | Gender contradictions                                                                                         |
| 10 | Student        | Malformed institutional emails                                                                                |
| 11 | Student        | Email collisions affecting multiple students                                                                  |
| 12 | Student        | Duplicate student records with conflicting ages                                                               |
| 13 | Student        | Additional missing full names                                                                                 |
| 14 | Student        | Negative age (-22)                                                                                            |
| 15 | Student        | Negative age (-5)                                                                                             |
| 16 | Student        | Additional gender encoding inconsistencies                                                                    |
| 17 | Student        | Unrealistic age (4)                                                                                           |
| 18 | Student        | Unrealistic age (121)                                                                                         |
| 19 | Student        | Invalid email format (@kayfa.io)                                                                              |
| 20 | Student        | Non-existent groups (GZZ, G77)                                                                                |
| 21 | Student        | Student identity duplication through shared emails                                                            |
| 22 | Student        | Further gender encoding inconsistencies                                                                       |
| 23 | Grades         | Incorrect max_score value (10 instead of 100)                                                                 |
| 24 | Grades         | score_pct outside valid range                                                                                 |
| 25 | Grades         | Inconsistent mastery_status                                                                                   |
| 26 | Engagement     | Negative engagement duration                                                                                  |
| 27 | Engagement     | Unrealistic engagement duration (99,999 sec)                                                                  |
| 28 | Engagement     | Orphan engagement record linked to non-existent student                                                       |
| 29 | Engagement     | Duplicate engagement event IDs                                                                                |
| 30 | Engagement     | Missing engagement durations                                                                                  |
| 31 | Submissions    | Incorrect is_late flag                                                                                        |
| 32 | Submissions    | Negative time_spent_minutes                                                                                   |
| 33 | Submissions    | Missing submitted_at timestamp                                                                                |
| 34 | Submissions    | Duplicate submission records                                                                                  |
| 35 | Concepts       | Duplicate concept record_ids                                                                                  |
| 36 | Concepts       | Missing num_failed_concepts after merge                                                                       |
| 37 | Master Dataset | Post-merge/master dataset validation issue (cross-dataset integrity or missing values introduced after joins) |
